In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"

OUTPUT_ROOT = "/kaggle/working/mipaas_full"
PROCESSED_DIR = os.path.join(OUTPUT_ROOT, "processed_npy")

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("Output:", OUTPUT_ROOT)
print("Processed:", PROCESSED_DIR)

DATA_ROOT: /kaggle/input/datasets/nih-chest-xrays/data
Output: /kaggle/working/mipaas_full
Processed: /kaggle/working/mipaas_full/processed_npy


In [2]:
metadata_path = os.path.join(
    DATA_ROOT,
    "Data_Entry_2017.csv"
)

df = pd.read_csv(metadata_path)

print("Full metadata:", df.shape)
print(df.columns.tolist())

Full metadata: (112120, 12)
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']


In [3]:
DISEASES = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

for disease in DISEASES:
    df[disease] = df["Finding Labels"].apply(
        lambda x: int(disease in x.split("|"))
    )

print(df[DISEASES].sum())

Atelectasis           11559
Cardiomegaly           2776
Consolidation          4667
Edema                  2303
Effusion              13317
Emphysema              2516
Fibrosis               1686
Hernia                  227
Infiltration          19894
Mass                   5782
Nodule                 6331
Pleural_Thickening     3385
Pneumonia              1431
Pneumothorax           5302
dtype: int64


In [4]:
with open(
    os.path.join(DATA_ROOT, "train_val_list.txt")
) as f:
    trainval_images = [x.strip() for x in f]

with open(
    os.path.join(DATA_ROOT, "test_list.txt")
) as f:
    official_test_images = [x.strip() for x in f]

print("Official train/validation images:", len(trainval_images))
print("Official test images:", len(official_test_images))

Official train/validation images: 86524
Official test images: 25596


In [5]:
trainval_df = df[
    df["Image Index"].isin(trainval_images)
].copy()

official_test_df = df[
    df["Image Index"].isin(official_test_images)
].copy()

print("Train/validation:", trainval_df.shape)
print("Official test:", official_test_df.shape)

Train/validation: (86524, 26)
Official test: (25596, 26)


In [6]:
overlap = set(trainval_df["Image Index"]) & set(
    official_test_df["Image Index"]
)

print("Image overlap:", len(overlap))

Image overlap: 0


In [7]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        trainval_df,
        groups=trainval_df["Patient ID"]
    )
)

train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

print("Train images:", len(train_df))
print("Validation images:", len(val_df))

print(
    "Train patients:",
    train_df["Patient ID"].nunique()
)

print(
    "Validation patients:",
    val_df["Patient ID"].nunique()
)

Train images: 77988
Validation images: 8536
Train patients: 25207
Validation patients: 2801


In [8]:
patient_overlap = (
    set(train_df["Patient ID"])
    &
    set(val_df["Patient ID"])
)

print("Patient overlap:", len(patient_overlap))

Patient overlap: 0


In [9]:
print("========== TRAIN ==========")
print(train_df[DISEASES].sum())

print("\n========== VALIDATION ==========")
print(val_df[DISEASES].sum())

print("\n========== OFFICIAL TEST ==========")
print(official_test_df[DISEASES].sum())

========== TRAIN ==========
Atelectasis            7405
Cardiomegaly           1549
Consolidation          2584
Edema                  1253
Effusion               7852
Emphysema              1317
Fibrosis               1111
Hernia                  122
Infiltration          12480
Mass                   3646
Nodule                 4247
Pleural_Thickening     1997
Pneumonia               782
Pneumothorax           2394
dtype: int64

========== VALIDATION ==========
Atelectasis            875
Cardiomegaly           158
Consolidation          268
Edema                  125
Effusion               807
Emphysema              106
Fibrosis               140
Hernia                  19
Infiltration          1302
Mass                   388
Nodule                 461
Pleural_Thickening     245
Pneumonia               94
Pneumothorax           243
dtype: int64

========== OFFICIAL TEST ==========
Atelectasis           3279
Cardiomegaly          1069
Consolidation         1815
Edema                  9

In [10]:
import os
from tqdm.auto import tqdm

image_path_map = {}

for i in range(1, 13):

    folder = os.path.join(
        DATA_ROOT,
        f"images_{i:03d}",
        "images"
    )

    print(f"Indexing images_{i:03d}...")

    for entry in os.scandir(folder):

        if (
            entry.is_file()
            and entry.name.lower().endswith(".png")
        ):
            image_path_map[entry.name] = entry.path

print("\nTotal images indexed:", len(image_path_map))

Indexing images_001...
Indexing images_002...
Indexing images_003...
Indexing images_004...
Indexing images_005...
Indexing images_006...
Indexing images_007...
Indexing images_008...
Indexing images_009...
Indexing images_010...
Indexing images_011...
Indexing images_012...

Total images indexed: 112120


In [11]:
all_required = set(df["Image Index"])

missing = all_required - set(image_path_map.keys())

print("Missing images:", len(missing))

if missing:
    print("First missing:", list(missing)[:10])

Missing images: 0


In [12]:
import os
import cv2
import numpy as np
from tqdm.auto import tqdm

PROCESSED_DIR = os.path.join(
    OUTPUT_ROOT,
    "processed_npy"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

failed_images = []

for image_name, input_path in tqdm(
    image_path_map.items(),
    total=len(image_path_map),
    desc="Preprocessing all X-rays"
):

    output_path = os.path.join(
        PROCESSED_DIR,
        image_name.replace(".png", ".npy")
    )

    # Skip files already processed
    if os.path.exists(output_path):
        continue

    image = cv2.imread(
        input_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        failed_images.append(image_name)
        continue

    # CLAHE
    image = clahe.apply(image)

    # Resize
    image = cv2.resize(
        image,
        (224, 224),
        interpolation=cv2.INTER_AREA
    )

    # Save as uint8 to keep storage reasonable
    np.save(
        output_path,
        image.astype(np.uint8)
    )

print("\n========== PREPROCESSING COMPLETE ==========")
print("Processed directory:", PROCESSED_DIR)
print("Failed images:", len(failed_images))

if failed_images:
    print("First failures:", failed_images[:10])

Preprocessing all X-rays:   0%|          | 0/112120 [00:00<?, ?it/s]

libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG
libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG



========== PREPROCESSING COMPLETE ==========
Processed directory: /kaggle/working/mipaas_full/processed_npy
Failed images: 0


In [13]:
import os
import numpy as np

PROCESSED_DIR = "/kaggle/working/mipaas_full/processed_npy"

processed_files = os.listdir(PROCESSED_DIR)

print("Cached files:", len(processed_files))
print("Expected:", 112120)

sample_file = processed_files[0]
arr = np.load(os.path.join(PROCESSED_DIR, sample_file))

print("Sample:", sample_file)
print("Shape:", arr.shape)
print("Dtype:", arr.dtype)
print("Min:", arr.min())
print("Max:", arr.max())

Cached files: 112120
Expected: 112120
Sample: 00019130_000.npy
Shape: (224, 224)
Dtype: uint8
Min: 1
Max: 251


In [14]:
def add_processed_path(df):
    df = df.copy()

    df["processed_path"] = df["Image Index"].apply(
        lambda x: os.path.join(
            PROCESSED_DIR,
            x.replace(".png", ".npy")
        )
    )

    return df


train_df = add_processed_path(train_df)
val_df = add_processed_path(val_df)
official_test_df = add_processed_path(official_test_df)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Official test:", official_test_df.shape)

Train: (77988, 27)
Validation: (8536, 27)
Official test: (25596, 27)


In [15]:
print(
    "Missing train:",
    (~train_df["processed_path"].map(os.path.exists)).sum()
)

print(
    "Missing validation:",
    (~val_df["processed_path"].map(os.path.exists)).sum()
)

print(
    "Missing test:",
    (~official_test_df["processed_path"].map(os.path.exists)).sum()
)

Missing train: 0
Missing validation: 0
Missing test: 0


In [16]:
import os
import json

METADATA_DIR = "/kaggle/working/mipaas_full/metadata"
os.makedirs(METADATA_DIR, exist_ok=True)

train_df.to_csv(
    os.path.join(METADATA_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(METADATA_DIR, "val.csv"),
    index=False
)

official_test_df.to_csv(
    os.path.join(METADATA_DIR, "official_test.csv"),
    index=False
)

config = {
    "dataset": "NIH ChestX-ray14",

    "total_images": 112120,
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "official_test_images": len(official_test_df),

    "split": {
        "method": "GroupShuffleSplit",
        "validation_size": 0.10,
        "random_state": 42,
        "group": "Patient ID"
    },

    "preprocessing": {
        "grayscale": True,
        "clahe": True,
        "clahe_clip_limit": 2.0,
        "clahe_tile_grid_size": [8, 8],
        "resize": [224, 224],
        "dtype": "uint8",
        "format": "npy"
    },

    "labels": DISEASES
}

with open(
    os.path.join(METADATA_DIR, "dataset_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=4)

print("Metadata saved successfully!")

Metadata saved successfully!


In [17]:
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])

print("Patient overlap:", len(train_patients & val_patients))

Patient overlap: 0


In [18]:
import os

os.makedirs("/kaggle/working/emergency_backup", exist_ok=True)

train_df.to_pickle("/kaggle/working/emergency_backup/train_df.pkl")
val_df.to_pickle("/kaggle/working/emergency_backup/val_df.pkl")

train_df.to_csv("/kaggle/working/emergency_backup/train_df.csv", index=False)
val_df.to_csv("/kaggle/working/emergency_backup/val_df.csv", index=False)

print("BACKUP SAVED")
print("train:", train_df.shape)
print("val:", val_df.shape)

BACKUP SAVED
train: (77988, 27)
val: (8536, 27)


In [19]:
import os
import shutil

backup_dir = "/kaggle/working/emergency_backup"

# Save the current dataframes
train_df.to_pickle(f"{backup_dir}/train_df.pkl")
val_df.to_pickle(f"{backup_dir}/val_df.pkl")

train_df.to_csv(f"{backup_dir}/train_df.csv", index=False)
val_df.to_csv(f"{backup_dir}/val_df.csv", index=False)

# Create one ZIP containing everything
zip_path = shutil.make_archive(
    "/kaggle/working/emergency_backup",
    "zip",
    backup_dir
)

print("✅ EMERGENCY BACKUP CREATED!")
print(zip_path)

# Show file sizes
for root, dirs, files in os.walk(backup_dir):
    for file in files:
        path = os.path.join(root, file)
        print(f"{file}: {os.path.getsize(path) / (1024**2):.2f} MB")

✅ EMERGENCY BACKUP CREATED!
/kaggle/working/emergency_backup.zip
train_df.csv: 11.62 MB
train_df.pkl: 20.78 MB
val_df.csv: 1.27 MB
val_df.pkl: 2.28 MB


In [20]:
from IPython.display import FileLink

FileLink("/kaggle/working/emergency_backup.zip")

/kaggle/working/emergency_backup.zip

In [21]:
import os

base = "/kaggle/working"

for name in os.listdir(base):
    path = os.path.join(base, name)

    if os.path.isdir(path):
        total = 0
        for root, dirs, files in os.walk(path):
            for f in files:
                try:
                    total += os.path.getsize(os.path.join(root, f))
                except:
                    pass
        print(f"📁 {name}: {total / (1024**3):.2f} GB")

    else:
        try:
            size = os.path.getsize(path)
            print(f"📄 {name}: {size / (1024**3):.2f} GB")
        except:
            pass

📁 mipaas_full: 5.27 GB
📄 emergency_backup.zip: 0.00 GB
📁 .virtual_documents: 0.00 GB
📁 emergency_backup: 0.04 GB


In [22]:
import os

base = "/kaggle/working/mipaas_full"

for name in os.listdir(base):
    path = os.path.join(base, name)

    if os.path.isdir(path):
        total = 0
        count = 0

        for root, dirs, files in os.walk(path):
            for f in files:
                try:
                    total += os.path.getsize(os.path.join(root, f))
                    count += 1
                except:
                    pass

        print(f"📁 {name}: {total / (1024**3):.2f} GB ({count:,} files)")

    else:
        size = os.path.getsize(path)
        print(f"📄 {name}: {size / (1024**3):.2f} GB")

📁 processed_npy: 5.25 GB (112,120 files)
📁 metadata: 0.02 GB (4 files)


In [23]:
   !du -sh /kaggle/working; find /kaggle/working -type f | wc -l

5.7G	/kaggle/working
112130


In [24]:
import os
fs = sorted(os.listdir("/kaggle/working"))
print(len(fs)); print(fs[:5]); print(fs[-5:])
print([f for f in fs if not f.endswith(".npy")])

4
['.virtual_documents', 'emergency_backup', 'emergency_backup.zip', 'mipaas_full']
['.virtual_documents', 'emergency_backup', 'emergency_backup.zip', 'mipaas_full']
['.virtual_documents', 'emergency_backup', 'emergency_backup.zip', 'mipaas_full']


In [25]:
import os, numpy as np
SRC = "/kaggle/working/mipaas_full/processed_npy"
files = sorted(f for f in os.listdir(SRC) if f.endswith(".npy"))
print(len(files), files[:3])
a = np.load(f"{SRC}/{files[0]}")
print(a.dtype, a.shape, a.min(), a.max())

112120 ['00000001_000.npy', '00000001_001.npy', '00000001_002.npy']
uint8 (224, 224) 1 254


In [26]:
import pandas as pd
DST = "/kaggle/working/mipaas_full/shards"
os.makedirs(DST, exist_ok=True)
SHARD, index = 10000, []

for sid, start in enumerate(range(0, len(files), SHARD)):
    batch = files[start:start + SHARD]
    arr = np.stack([np.load(f"{SRC}/{f}") for f in batch]).astype(np.uint8)
    np.save(f"{DST}/shard_{sid:02d}.npy", arr)
    index += [(f, sid, i) for i, f in enumerate(batch)]
    print(sid, arr.shape, flush=True)

pd.DataFrame(index, columns=["file", "shard", "row"]).to_csv(f"{DST}/shard_index.csv", index=False)

0 (10000, 224, 224)
1 (10000, 224, 224)
2 (10000, 224, 224)
3 (10000, 224, 224)
4 (10000, 224, 224)
5 (10000, 224, 224)
6 (10000, 224, 224)
7 (10000, 224, 224)
8 (10000, 224, 224)
9 (10000, 224, 224)
10 (10000, 224, 224)
11 (2120, 224, 224)


In [27]:
n = sum(len(np.load(f"{DST}/{f}", mmap_mode="r")) for f in os.listdir(DST) if f.endswith(".npy"))
print(n, len(files))
# spot check: shard row must equal the original file
chk = np.load(f"{DST}/shard_03.npy", mmap_mode="r")[5]
print(np.array_equal(chk, np.load(f"{SRC}/{files[30005]}")))

112120 112120
True


In [28]:
import shutil
shutil.rmtree(SRC)
!find /kaggle/working -type f | wc -l

23


In [31]:
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()

token = s.get_secret("MIPAAS")

print("Token found:", bool(token))
print("Token length:", len(token) if token else 0)

Token found: True
Token length: 37


In [33]:
import os
import json
import shutil
from kaggle_secrets import UserSecretsClient

# Get secure API token
s = UserSecretsClient()
token = s.get_secret("MIPAAS")
os.environ["MIPAAS"] = token

# Existing processed data — DO NOT regenerate
up = "/kaggle/working/mipaas_full"

# Add our emergency dataframe backup inside the dataset
shutil.copytree(
    "/kaggle/working/emergency_backup",
    f"{up}/split",
    dirs_exist_ok=True
)

# Dataset metadata
metadata = {
    "title": "mipaas-full-preprocessed",
    "id": "shreyagupta1181/mipaas-full-preprocessed",
    "licenses": [
        {"name": "CC0-1.0"}
    ]
}

with open(f"{up}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Dataset prepared")
print(f"📁 {up}")
print("📦 Existing processed_npy will be uploaded — no preprocessing required.")

✅ Dataset prepared
📁 /kaggle/working/mipaas_full
📦 Existing processed_npy will be uploaded — no preprocessing required.


In [32]:
import os
print(os.environ.get("KAGGLE_USERNAME", "not set"))

not set


In [34]:
!kaggle datasets create -p /kaggle/working/mipaas_full -r zip

Starting upload for file shards.zip
100%|██████████████████████████████████████| 4.39G/4.39G [00:57<00:00, 81.6MB/s]
Upload successful: shards.zip (4GB)
Starting upload for file metadata.zip
100%|██████████████████████████████████████| 1.43M/1.43M [00:00<00:00, 2.39MB/s]
Upload successful: metadata.zip (1MB)
Starting upload for file split.zip
100%|██████████████████████████████████████| 2.22M/2.22M [00:00<00:00, 3.01MB/s]
Upload successful: split.zip (2MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/shreyagupta1181/mipaas-full-preprocessed


In [8]:
import os

print("=== /kaggle/input ===")
print(os.listdir("/kaggle/input"))

print("\n=== Walking /kaggle/input ===")
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:5])
    if root.count(os.sep) > 4:
        dirs[:] = []

=== /kaggle/input ===
['datasets']

=== Walking /kaggle/input ===
/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/shreyagupta1181 []
/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed []
/kaggle/input/datasets/nih-chest-xrays []
/kaggle/input/datasets/nih-chest-xrays/data ['LOG_CHESTXRAY.pdf', 'README_CHESTXRAY.pdf', 'BBox_List_2017.csv', 'test_list.txt', 'ARXIV_V5_CHESTXRAY.pdf']


In [9]:
import os

base = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

print("Exists:", os.path.exists(base))
print("Is directory:", os.path.isdir(base))
print("Contents:", os.listdir(base))

print("\nDetailed listing:")
os.system(f"ls -lah '{base}'")

Exists: True
Is directory: True
Contents: ['metadata', 'split', 'shards']

Detailed listing:
total 4.0K
drwxr-xr-x 5 nobody nogroup    0 Sep 29 13:35 .
drwxr-xr-x 3 root   root    4.0K Sep 29 13:36 ..
drwxr-xr-x 2 nobody nogroup    0 Sep 29 13:35 metadata
drwxr-xr-x 2 nobody nogroup    0 Sep 29 13:35 shards
drwxr-xr-x 2 nobody nogroup    0 Sep 29 13:35 split


0

In [10]:
import os

base = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

for folder in ["metadata", "split", "shards"]:
    path = os.path.join(base, folder)
    files = os.listdir(path)

    print(f"\n📁 {folder}: {len(files)} files")
    print(files[:15])


📁 metadata: 4 files
['official_test.csv', 'val.csv', 'dataset_config.json', 'train.csv']

📁 split: 4 files
['val_df.csv', 'train_df.pkl', 'train_df.csv', 'val_df.pkl']

📁 shards: 13 files
['shard_01.npy', 'shard_07.npy', 'shard_09.npy', 'shard_05.npy', 'shard_02.npy', 'shard_index.csv', 'shard_06.npy', 'shard_04.npy', 'shard_10.npy', 'shard_03.npy', 'shard_08.npy', 'shard_11.npy', 'shard_00.npy']


In [11]:
import numpy as np
import os

base = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed/shards"

shard = np.load(
    os.path.join(base, "shard_00.npy"),
    mmap_mode="r"
)

print("Shape:", shard.shape)
print("Dtype:", shard.dtype)
print("Min:", shard.min())
print("Max:", shard.max())
print("Memory mapped:", type(shard))

Shape: (10000, 224, 224)
Dtype: uint8
Min: 0
Max: 255
Memory mapped: <class 'numpy.memmap'>


In [12]:
import pandas as pd
import os

base = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

index_path = os.path.join(base, "shards", "shard_index.csv")

idx = pd.read_csv(index_path)

print("Index shape:", idx.shape)
print("\nColumns:", idx.columns.tolist())
print("\nFirst 5 rows:")
print(idx.head())
print("\nLast 5 rows:")
print(idx.tail())

Index shape: (112120, 3)

Columns: ['file', 'shard', 'row']

First 5 rows:
               file  shard  row
0  00000001_000.npy      0    0
1  00000001_001.npy      0    1
2  00000001_002.npy      0    2
3  00000002_000.npy      0    3
4  00000003_000.npy      0    4

Last 5 rows:
                    file  shard   row
112115  00030801_001.npy     11  2115
112116  00030802_000.npy     11  2116
112117  00030803_000.npy     11  2117
112118  00030804_000.npy     11  2118
112119  00030805_000.npy     11  2119


In [13]:
train_df = pd.read_csv(os.path.join(base, "split", "train_df.csv"))
val_df   = pd.read_csv(os.path.join(base, "split", "val_df.csv"))

print("Train:", train_df.shape)
print("Val:", val_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nVal columns:")
print(val_df.columns.tolist())

Train: (77988, 27)
Val: (8536, 27)

Train columns:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'processed_path']

Val columns:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'processed_path']


In [14]:
print("Train Image Index examples:")
print(train_df["Image Index"].head(10).tolist())

print("\nShard file examples:")
print(idx["file"].head(10).tolist())

Train Image Index examples:
['00000001_000.png', '00000001_001.png', '00000001_002.png', '00000002_000.png', '00000004_000.png', '00000005_000.png', '00000005_001.png', '00000005_002.png', '00000005_003.png', '00000005_004.png']

Shard file examples:
['00000001_000.npy', '00000001_001.npy', '00000001_002.npy', '00000002_000.npy', '00000003_000.npy', '00000003_001.npy', '00000003_002.npy', '00000003_003.npy', '00000003_004.npy', '00000003_005.npy']


In [15]:
train_df["processed_file"] = (
    train_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

In [16]:
import os
import numpy as np
import pandas as pd

BASE = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"
SHARD_DIR = os.path.join(BASE, "shards")

# Load split data
train_df = pd.read_csv(os.path.join(BASE, "split", "train_df.csv"))
val_df = pd.read_csv(os.path.join(BASE, "split", "val_df.csv"))

# Load shard index
shard_idx = pd.read_csv(os.path.join(SHARD_DIR, "shard_index.csv"))

# Match the filename format
train_df["processed_file"] = (
    train_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

val_df["processed_file"] = (
    val_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

# Add shard + row information
train_df = train_df.merge(
    shard_idx,
    left_on="processed_file",
    right_on="file",
    how="left",
    validate="one_to_one"
)

val_df = val_df.merge(
    shard_idx,
    left_on="processed_file",
    right_on="file",
    how="left",
    validate="one_to_one"
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)

print("\nMissing train mappings:", train_df["shard"].isna().sum())
print("Missing val mappings:", val_df["shard"].isna().sum())

print("\nExample mapping:")
print(train_df[["Image Index", "shard", "row"]].head())

Train: (77988, 31)
Val: (8536, 31)

Missing train mappings: 0
Missing val mappings: 0

Example mapping:
        Image Index  shard  row
0  00000001_000.png      0    0
1  00000001_001.png      0    1
2  00000001_002.png      0    2
3  00000002_000.png      0    3
4  00000004_000.png      0   12


In [17]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SHARD_DIR = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed/shards"

# The 15 disease labels in your dataframe
LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

# Cache the shard arrays as memory maps.
# This DOES NOT load the 4+ GB into RAM.
shards = {}

def get_shard(shard_id):
    shard_id = int(shard_id)

    if shard_id not in shards:
        path = os.path.join(
            SHARD_DIR,
            f"shard_{shard_id:02d}.npy"
        )
        shards[shard_id] = np.load(path, mmap_mode="r")

    return shards[shard_id]


class ChestXrayDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Retrieve one image from the correct shard
        shard = get_shard(row["shard"])
        image = shard[int(row["row"])]

        # uint8 [0,255] -> float32 [0,1]
        image = torch.from_numpy(
            image.copy()
        ).float() / 255.0

        # Grayscale: H,W -> 1,H,W
        image = image.unsqueeze(0)

        # Multi-label target
        target = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return image, target


# Create datasets
train_dataset = ChestXrayDataset(train_df)
val_dataset = ChestXrayDataset(val_df)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

# Start conservatively with batch size 64
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("DataLoaders created!")

Train samples: 77988
Val samples: 8536
DataLoaders created!


In [18]:
images, targets = next(iter(train_loader))

print("Images:", images.shape)
print("Targets:", targets.shape)
print("Image dtype:", images.dtype)
print("Image range:", images.min().item(), images.max().item())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

images = images.to(device, non_blocking=True)
targets = targets.to(device, non_blocking=True)

print("GPU batch:", images.shape)

Images: torch.Size([64, 1, 224, 224])
Targets: torch.Size([64, 14])
Image dtype: torch.float32
Image range: 0.003921568859368563 1.0
Device: cuda
GPU batch: torch.Size([64, 1, 224, 224])


In [19]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pretrained ResNet-18
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Change RGB input (3 channels) -> grayscale (1 channel)
old_conv = model.conv1

model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# Initialize grayscale conv from pretrained RGB weights
with torch.no_grad():
    model.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

# 14 disease labels
model.fc = nn.Linear(
    model.fc.in_features,
    len(LABELS)
)

model = model.to(device)

print(model)
print("\nDevice:", next(model.parameters()).device)
print("Output classes:", len(LABELS))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]


ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [20]:
import torch

model.eval()

with torch.no_grad():
    outputs = model(images)

print("Input:", images.shape)
print("Output:", outputs.shape)
print("Output dtype:", outputs.dtype)
print("Output device:", outputs.device)
print("Sample logits:", outputs[0])

Input: torch.Size([64, 1, 224, 224])
Output: torch.Size([64, 14])
Output dtype: torch.float32
Output device: cuda:0
Sample logits: tensor([ 0.0914,  0.0381, -0.6400,  0.0931, -0.1442, -0.2529, -1.3459, -0.7452,
        -0.9155, -0.5527,  0.1591, -0.1748,  0.3516, -0.6764], device='cuda:0')


In [21]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# -----------------------------
# Training configuration
# -----------------------------
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Multi-label classification
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Mixed precision
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_val_loss = float("inf")
patience = 2
epochs_without_improvement = 0

print("Training on:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LR)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


# -----------------------------
# Training loop
# -----------------------------
for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0.0
    train_count = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]"
    )

    for images, targets in train_bar:

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        train_loss += loss.item() * batch_size
        train_count += batch_size

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss /= train_count


    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0
    val_count = 0

    with torch.no_grad():

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]"
        )

        for images, targets in val_bar:

            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, targets)

            batch_size = images.size(0)
            val_loss += loss.item() * batch_size
            val_count += batch_size

    val_loss /= val_count


    # =========================
    # RESULTS
    # =========================
    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "labels": LABELS,
            },
            "/kaggle/working/best_resnet18.pth"
        )

        print("✅ Best model saved!")

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

        if epochs_without_improvement >= patience:
            print("\n🛑 Early stopping.")
            break


print("\n🎉 TRAINING COMPLETE")
print("Best validation loss:", best_val_loss)

Training on: cuda
Epochs: 5
Learning rate: 0.0001
Train batches: 1219
Val batches: 134


Epoch 1/5 [TRAIN]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 1/5 [VAL]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1/5 | Train Loss: 0.1523 | Val Loss: 0.1363
✅ Best model saved!


Epoch 2/5 [TRAIN]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 2/5 [VAL]:   0%|          | 0/134 [00:00<?, ?it/s]

... (3 outputs left out)


Epoch 3/5 | Train Loss: 0.1207 | Val Loss: 0.1373
No improvement (1/2)


Epoch 4/5 [TRAIN]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 4/5 [VAL]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 4/5 | Train Loss: 0.1038 | Val Loss: 0.1508
No improvement (2/2)

🛑 Early stopping.

🎉 TRAINING COMPLETE
Best validation loss: 0.13327731739558565


In [22]:
import torch

checkpoint = torch.load(
    "/kaggle/working/best_resnet18.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best checkpoint")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])

Loaded best checkpoint
Best epoch: 2
Best validation loss: 0.13327731739558565


In [23]:
import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

model.eval()

all_targets = []
all_probs = []

with torch.no_grad():

    for images, targets in tqdm(
        val_loader,
        desc="Evaluating validation set"
    ):

        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        # Convert logits → probabilities
        probs = torch.sigmoid(outputs)

        all_targets.append(targets.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

# Combine all batches
y_true = np.concatenate(all_targets, axis=0)
y_prob = np.concatenate(all_probs, axis=0)

print("y_true shape:", y_true.shape)
print("y_prob shape:", y_prob.shape)

Evaluating validation set:   0%|          | 0/134 [00:00<?, ?it/s]

y_true shape: (8536, 14)
y_prob shape: (8536, 14)


In [24]:
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

results = []

for i, label in enumerate(LABELS):

    y_t = y_true[:, i]
    y_p = y_prob[:, i]

    # ROC-AUC
    if len(np.unique(y_t)) > 1:
        roc_auc = roc_auc_score(y_t, y_p)
    else:
        roc_auc = np.nan

    # PR-AUC / Average Precision
    if y_t.sum() > 0:
        pr_auc = average_precision_score(y_t, y_p)
    else:
        pr_auc = np.nan

    results.append({
        "Disease": label,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Positive Cases": int(y_t.sum())
    })

metrics_df = pd.DataFrame(results)

print(metrics_df.to_string(index=False))

print("\n" + "=" * 60)
print(f"Macro ROC-AUC: {metrics_df['ROC-AUC'].mean():.4f}")
print(f"Macro PR-AUC:  {metrics_df['PR-AUC'].mean():.4f}")
print("=" * 60)

           Disease  ROC-AUC   PR-AUC  Positive Cases
       Atelectasis 0.803510 0.338206             875
      Cardiomegaly 0.880086 0.189950             158
     Consolidation 0.811948 0.146095             268
             Edema 0.912424 0.142330             125
          Effusion 0.898782 0.525711             807
         Emphysema 0.860208 0.158092             106
          Fibrosis 0.792387 0.118521             140
            Hernia 0.813939 0.013423              19
      Infiltration 0.716573 0.326885            1302
              Mass 0.830246 0.280230             388
            Nodule 0.719850 0.200165             461
Pleural_Thickening 0.790875 0.149451             245
         Pneumonia 0.693303 0.037335              94
      Pneumothorax 0.861633 0.215603             243

Macro ROC-AUC: 0.8133
Macro PR-AUC:  0.2030


In [25]:
for i, label in enumerate(LABELS):
    positives = int(y_true[:, i].sum())
    total = len(y_true)
    prevalence = positives / total

    print(
        f"{label:20s} "
        f"positives={positives:5d} "
        f"prevalence={prevalence:.4%} "
        f"PR-AUC={metrics_df.loc[i, 'PR-AUC']:.4f}"
    )

Atelectasis          positives=  875 prevalence=10.2507% PR-AUC=0.3382
Cardiomegaly         positives=  158 prevalence=1.8510% PR-AUC=0.1899
Consolidation        positives=  268 prevalence=3.1396% PR-AUC=0.1461
Edema                positives=  125 prevalence=1.4644% PR-AUC=0.1423
Effusion             positives=  807 prevalence=9.4541% PR-AUC=0.5257
Emphysema            positives=  106 prevalence=1.2418% PR-AUC=0.1581
Fibrosis             positives=  140 prevalence=1.6401% PR-AUC=0.1185
Hernia               positives=   19 prevalence=0.2226% PR-AUC=0.0134
Infiltration         positives= 1302 prevalence=15.2530% PR-AUC=0.3269
Mass                 positives=  388 prevalence=4.5455% PR-AUC=0.2802
Nodule               positives=  461 prevalence=5.4007% PR-AUC=0.2002
Pleural_Thickening   positives=  245 prevalence=2.8702% PR-AUC=0.1495
Pneumonia            positives=   94 prevalence=1.1012% PR-AUC=0.0373
Pneumothorax         positives=  243 prevalence=2.8468% PR-AUC=0.2156


In [26]:
metrics_df.to_csv(
    "/kaggle/working/resnet18_baseline_metrics.csv",
    index=False
)

print("Baseline saved.")

Baseline saved.


In [27]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained DenseNet-121
model = models.densenet121(
    weights=models.DenseNet121_Weights.DEFAULT
)

# Change first layer: RGB (3 channels) → grayscale (1 channel)
old_conv = model.features.conv0

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# Initialize grayscale weights by averaging pretrained RGB weights
with torch.no_grad():
    new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

model.features.conv0 = new_conv

# Change classifier: 1000 ImageNet classes → 14 diseases
model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)

model = model.to(device)

print(model.classifier)
print("Device:", device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 157MB/s] 

Linear(in_features=1024, out_features=14, bias=True)
Device: cuda


In [28]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("DenseNet-121 ready!")

DenseNet-121 ready!


In [29]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Loss
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Mixed precision
scaler = torch.amp.GradScaler("cuda")

# Training settings
num_epochs = 5
patience = 2

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

best_model_path = "/kaggle/working/best_densenet121.pth"

for epoch in range(num_epochs):

    # -------------------------
    # TRAIN
    # -------------------------
    model.train()
    running_train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs} [Train]"
    )

    for images, targets in train_bar:

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_train_loss += loss.item() * images.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_train_loss / len(train_loader.dataset)

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{num_epochs} [Val]"
        )

        for images, targets in val_bar:

            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(images)
                loss = criterion(outputs, targets)

            running_val_loss += loss.item() * images.size(0)

            val_bar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss = running_val_loss / len(val_loader.dataset)

    print(
        f"\nEpoch {epoch+1}: "
        f"Train Loss = {train_loss:.4f}, "
        f"Val Loss = {val_loss:.4f}"
    )

    # -------------------------
    # SAVE BEST MODEL
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            },
            best_model_path
        )

        print(f"✓ Best model saved: {best_model_path}")

    else:
        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

print("\nTraining complete!")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")

Epoch 1/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1: Train Loss = 0.1501, Val Loss = 0.1363
✓ Best model saved: /kaggle/working/best_densenet121.pth


Epoch 2/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 2/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 2: Train Loss = 0.1286, Val Loss = 0.1307
✓ Best model saved: /kaggle/working/best_densenet121.pth


... (2 outputs left out)


Epoch 3: Train Loss = 0.1207, Val Loss = 0.1328
No improvement (1/2)


Epoch 4/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 4/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 4: Train Loss = 0.1103, Val Loss = 0.1370
No improvement (2/2)
Early stopping.

Training complete!
Best epoch: 2
Best validation loss: 0.1307


In [30]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

# --------------------------------------------------
# 1. Load BEST DenseNet checkpoint
# --------------------------------------------------

checkpoint_path = "/kaggle/working/best_densenet121.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Loaded best DenseNet-121 checkpoint")
print(f"Best epoch: {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['val_loss']:.4f}")


# --------------------------------------------------
# 2. Run predictions on validation set
# --------------------------------------------------

all_targets = []
all_probs = []

with torch.no_grad():

    for images, targets in tqdm(
        val_loader,
        desc="Evaluating DenseNet-121"
    ):

        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        # Convert logits → probabilities
        probs = torch.sigmoid(outputs)

        all_targets.append(targets.cpu().numpy())
        all_probs.append(probs.cpu().numpy())


# --------------------------------------------------
# 3. Combine batches
# --------------------------------------------------

y_true = np.concatenate(all_targets, axis=0)
y_prob = np.concatenate(all_probs, axis=0)

print("\nPrediction shapes:")
print("y_true:", y_true.shape)
print("y_prob:", y_prob.shape)


# --------------------------------------------------
# 4. Disease names
# --------------------------------------------------

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]


# --------------------------------------------------
# 5. Calculate ROC-AUC + PR-AUC
# --------------------------------------------------

results = []

for i, disease in enumerate(disease_cols):

    y = y_true[:, i]
    p = y_prob[:, i]

    # ROC-AUC
    roc_auc = roc_auc_score(y, p)

    # PR-AUC
    pr_auc = average_precision_score(y, p)

    positives = int(y.sum())
    prevalence = y.mean()

    results.append({
        "Disease": disease,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Positive Cases": positives,
        "Prevalence": prevalence
    })


# --------------------------------------------------
# 6. Results table
# --------------------------------------------------

results_df = pd.DataFrame(results)

print("\nDenseNet-121 Results:\n")
display(results_df)


# --------------------------------------------------
# 7. Macro averages
# --------------------------------------------------

macro_roc_auc = results_df["ROC-AUC"].mean()
macro_pr_auc = results_df["PR-AUC"].mean()

print("\n" + "=" * 50)
print("DenseNet-121 SUMMARY")
print("=" * 50)

print(f"Macro ROC-AUC : {macro_roc_auc:.4f}")
print(f"Macro PR-AUC  : {macro_pr_auc:.4f}")
print("=" * 50)

Loaded best DenseNet-121 checkpoint
Best epoch: 2
Best validation loss: 0.1307


Evaluating DenseNet-121:   0%|          | 0/134 [00:00<?, ?it/s]


Prediction shapes:
y_true: (8536, 14)
y_prob: (8536, 14)

DenseNet-121 Results:



,Disease,ROC-AUC,PR-AUC,Positive Cases,Prevalence
0,Atelectasis,0.814339,0.366150,875,0.102507
1,Cardiomegaly,0.903944,0.225320,158,0.018510
2,Consolidation,0.812669,0.155536,268,0.031396
3,Edema,0.908170,0.156369,125,0.014644
4,Effusion,0.901722,0.535978,807,0.094541
5,Emphysema,0.869703,0.178944,106,0.012418
6,Fibrosis,0.810665,0.141018,140,0.016401
7,Hernia,0.842167,0.024878,19,0.002226
8,Infiltration,0.723009,0.328882,1302,0.152530
9,Mass,0.842077,0.315000,388,0.045455



DenseNet-121 SUMMARY
Macro ROC-AUC : 0.8273
Macro PR-AUC  : 0.2195


In [31]:
import torch
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)

# ============================================================
# 1. Load official test metadata
# ============================================================

test_csv = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed/metadata/official_test.csv"

test_df = pd.read_csv(test_csv)

print("Official test set:")
print(test_df.shape)
print(test_df.head())


# ============================================================
# 2. Disease columns
# ============================================================

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

print("\nNumber of disease labels:", len(disease_cols))


# ============================================================
# 3. Prepare Image Index for shard mapping
# ============================================================

test_df["npy_name"] = (
    test_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

# shard_index should already be loaded from our previous setup.
# If not, load it here.

shard_index_path = (
    "/kaggle/input/datasets/shreyagupta1181/"
    "mipaas-full-preprocessed/shards/shard_index.csv"
)

shard_index = pd.read_csv(shard_index_path)

print("\nShard index:")
print(shard_index.shape)


# ============================================================
# 4. Map test images → shard + row
# ============================================================

test_df = test_df.merge(
    shard_index,
    left_on="npy_name",
    right_on="file",
    how="left",
    validate="one_to_one"
)

missing = test_df["shard"].isna().sum()

print("\nMissing shard mappings:", missing)

if missing > 0:
    raise ValueError(
        f"{missing} test images could not be mapped to shards."
    )

test_df["shard"] = test_df["shard"].astype(int)
test_df["row"] = test_df["row"].astype(int)

print("Test mapping successful.")


# ============================================================
# 5. Test Dataset
# ============================================================

class TestShardDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        shard_id = int(row["shard"])
        row_id = int(row["row"])

        image = shards[shard_id][row_id]

        # uint8 → float32
        image = torch.from_numpy(
            image.astype(np.float32)
        ) / 255.0

        # Add channel dimension
        image = image.unsqueeze(0)

        # 14 disease labels
        target = torch.tensor(
            row[disease_cols].values.astype(np.float32)
        )

        return image, target


test_dataset = TestShardDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("\nTest dataset:", len(test_dataset))
print("Test batches:", len(test_loader))


# ============================================================
# 6. Load BEST DenseNet-121
# ============================================================

checkpoint_path = "/kaggle/working/best_densenet121.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("\nLoaded DenseNet-121")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])


# ============================================================
# 7. Generate test predictions
# ============================================================

all_targets = []
all_probs = []

with torch.no_grad():

    for images, targets in tqdm(
        test_loader,
        desc="Testing DenseNet-121"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        probs = torch.sigmoid(outputs)

        all_targets.append(
            targets.numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )


y_test = np.concatenate(
    all_targets,
    axis=0
)

y_prob_test = np.concatenate(
    all_probs,
    axis=0
)

print("\nPrediction shapes:")
print("y_test:", y_test.shape)
print("y_prob_test:", y_prob_test.shape)


# ============================================================
# 8. Threshold predictions
# ============================================================

THRESHOLD = 0.5

y_pred_test = (
    y_prob_test >= THRESHOLD
).astype(int)

y_true_test = y_test.astype(int)


# ============================================================
# 9. ROC-AUC and PR-AUC
# ============================================================

roc_auc_per_class = []
pr_auc_per_class = []

for i, disease in enumerate(disease_cols):

    roc = roc_auc_score(
        y_true_test[:, i],
        y_prob_test[:, i]
    )

    pr = average_precision_score(
        y_true_test[:, i],
        y_prob_test[:, i]
    )

    roc_auc_per_class.append(roc)
    pr_auc_per_class.append(pr)


macro_roc_auc = np.mean(
    roc_auc_per_class
)

macro_pr_auc = np.mean(
    pr_auc_per_class
)

micro_roc_auc = roc_auc_score(
    y_true_test.ravel(),
    y_prob_test.ravel()
)

micro_pr_auc = average_precision_score(
    y_true_test.ravel(),
    y_prob_test.ravel()
)


# ============================================================
# 10. Classification metrics @ threshold 0.5
# ============================================================

exact_match_accuracy = accuracy_score(
    y_true_test,
    y_pred_test
)

micro_accuracy = np.mean(
    y_true_test == y_pred_test
)

hamming = hamming_loss(
    y_true_test,
    y_pred_test
)

macro_precision = precision_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

micro_precision = precision_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)


# ============================================================
# 11. Per-disease metrics
# ============================================================

per_disease = []

for i, disease in enumerate(disease_cols):

    precision = precision_score(
        y_true_test[:, i],
        y_pred_test[:, i],
        zero_division=0
    )

    recall = recall_score(
        y_true_test[:, i],
        y_pred_test[:, i],
        zero_division=0
    )

    f1 = f1_score(
        y_true_test[:, i],
        y_pred_test[:, i],
        zero_division=0
    )

    positives = int(
        y_true_test[:, i].sum()
    )

    per_disease.append({
        "Disease": disease,
        "ROC-AUC": roc_auc_per_class[i],
        "PR-AUC": pr_auc_per_class[i],
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Positive Cases": positives
    })


per_disease_df = pd.DataFrame(
    per_disease
)


# ============================================================
# 12. Final baseline summary
# ============================================================

print("\n")
print("=" * 60)
print("FINAL DENSENET-121 BASELINE — OFFICIAL TEST SET")
print("=" * 60)

print(f"Test images:           {len(test_dataset)}")

print("\nRanking metrics:")
print(f"Macro ROC-AUC:         {macro_roc_auc:.4f}")
print(f"Micro ROC-AUC:         {micro_roc_auc:.4f}")
print(f"Macro PR-AUC:          {macro_pr_auc:.4f}")
print(f"Micro PR-AUC:          {micro_pr_auc:.4f}")

print("\nClassification @ threshold 0.5:")
print(f"Exact-match accuracy:  {exact_match_accuracy:.4f}")
print(f"Label-wise accuracy:   {micro_accuracy:.4f}")
print(f"Hamming loss:          {hamming:.4f}")

print("\nMacro:")
print(f"Precision:             {macro_precision:.4f}")
print(f"Recall:                {macro_recall:.4f}")
print(f"F1:                    {macro_f1:.4f}")

print("\nMicro:")
print(f"Precision:             {micro_precision:.4f}")
print(f"Recall:                {micro_recall:.4f}")
print(f"F1:                    {micro_f1:.4f}")

print("=" * 60)

print("\nPer-disease results:")
display(per_disease_df)

Official test set:
(25596, 27)
        Image Index       Finding Labels  Follow-up #  Patient ID  \
0  00000003_000.png               Hernia            0           3   
1  00000003_001.png               Hernia            1           3   
2  00000003_002.png               Hernia            2           3   
3  00000003_003.png  Hernia|Infiltration            3           3   
4  00000003_004.png               Hernia            4           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           81              F            PA                 2582     2991   
1           74              F            PA                 2500     2048   
2           75              F            PA                 2048     2500   
3           76              F            PA                 2698     2991   
4           77              F            PA                 2500     2048   

   OriginalImagePixelSpacing[x  ...  Emphysema  Fibrosis  Hernia  \
0                      

Testing DenseNet-121:   0%|          | 0/400 [00:00<?, ?it/s]

KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_58/2502975020.py", line 124, in __getitem__
    image = shards[shard_id][row_id]
            ~~~~~~^^^^^^^^^^
KeyError: 0


In [32]:
DATASET_ROOT = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"
SHARD_DIR = f"{DATASET_ROOT}/shards"

print(SHARD_DIR)

/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed/shards


In [33]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SHARD_DIR = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed/shards"

class TestShardDataset(Dataset):

    def __init__(self, df, shard_dir):
        self.df = df.reset_index(drop=True)
        self.shard_dir = shard_dir
        self.shards = {}

    def __len__(self):
        return len(self.df)

    def get_shard(self, shard_id):

        if shard_id not in self.shards:

            shard_path = os.path.join(
                self.shard_dir,
                f"shard_{shard_id:02d}.npy"
            )

            self.shards[shard_id] = np.load(
                shard_path,
                mmap_mode="r"
            )

        return self.shards[shard_id]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        shard_id = int(row["shard"])
        row_id = int(row["row"])

        shard = self.get_shard(shard_id)

        image = shard[row_id]

        image = torch.from_numpy(
            image.astype(np.float32)
        ) / 255.0

        image = image.unsqueeze(0)

        target = torch.tensor(
            row[disease_cols].values.astype(np.float32)
        )

        return image, target


test_dataset = TestShardDataset(
    test_df,
    SHARD_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Test dataset:", len(test_dataset))
print("Test batches:", len(test_loader))

Test dataset: 25596
Test batches: 400


In [34]:
images, targets = next(iter(test_loader))

print("Images:", images.shape)
print("Targets:", targets.shape)
print("Image dtype:", images.dtype)
print("Target dtype:", targets.dtype)
print("Image range:", images.min().item(), images.max().item())

Images: torch.Size([64, 1, 224, 224])
Targets: torch.Size([64, 14])
Image dtype: torch.float32
Target dtype: torch.float32
Image range: 0.003921568859368563 1.0


In [35]:
# ============================================================
# FINAL TEST EVALUATION — DenseNet-121
# ============================================================

import torch
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

# ------------------------------------------------------------
# 1. Load BEST DenseNet-121 checkpoint
# ------------------------------------------------------------

checkpoint_path = "/kaggle/working/best_densenet121.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded best DenseNet-121 checkpoint")
print("Best epoch:", checkpoint["epoch"])
print(f"Best validation loss: {checkpoint['val_loss']:.4f}")


# ------------------------------------------------------------
# 2. Generate predictions on OFFICIAL TEST SET
# ------------------------------------------------------------

all_targets = []
all_probs = []

with torch.no_grad():

    for images, targets in tqdm(
        test_loader,
        desc="FINAL TEST — DenseNet-121"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        # Logits → probabilities
        probs = torch.sigmoid(outputs)

        all_targets.append(
            targets.cpu().numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )


# ------------------------------------------------------------
# 3. Combine predictions
# ------------------------------------------------------------

y_test = np.concatenate(
    all_targets,
    axis=0
)

y_prob_test = np.concatenate(
    all_probs,
    axis=0
)

y_true_test = y_test.astype(int)

print("\nPrediction shapes:")
print("y_true:", y_true_test.shape)
print("y_prob:", y_prob_test.shape)


# ------------------------------------------------------------
# 4. Disease names
# ------------------------------------------------------------

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]


# ------------------------------------------------------------
# 5. ROC-AUC + PR-AUC
# ------------------------------------------------------------

per_disease_results = []

roc_values = []
pr_values = []

for i, disease in enumerate(disease_cols):

    y_true = y_true_test[:, i]
    y_prob = y_prob_test[:, i]

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    roc_values.append(roc_auc)
    pr_values.append(pr_auc)

    per_disease_results.append({
        "Disease": disease,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Positive Cases": int(y_true.sum()),
        "Prevalence": y_true.mean()
    })


# ------------------------------------------------------------
# 6. Macro / Micro ranking metrics
# ------------------------------------------------------------

macro_roc_auc = np.mean(roc_values)
macro_pr_auc = np.mean(pr_values)

micro_roc_auc = roc_auc_score(
    y_true_test.ravel(),
    y_prob_test.ravel()
)

micro_pr_auc = average_precision_score(
    y_true_test.ravel(),
    y_prob_test.ravel()
)


# ------------------------------------------------------------
# 7. Threshold-based metrics
# ------------------------------------------------------------
# 0.5 is the standard baseline threshold.
# We will NOT tune this on the test set.

THRESHOLD = 0.5

y_pred_test = (
    y_prob_test >= THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 8. Overall classification metrics
# ------------------------------------------------------------

# Exact-match:
# Entire 14-label prediction must match the ground truth.
exact_match_accuracy = accuracy_score(
    y_true_test,
    y_pred_test
)

# Label-wise accuracy:
# Percentage of all individual disease labels classified correctly.
label_wise_accuracy = np.mean(
    y_true_test == y_pred_test
)

hamming = hamming_loss(
    y_true_test,
    y_pred_test
)

macro_precision = precision_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true_test,
    y_pred_test,
    average="macro",
    zero_division=0
)

micro_precision = precision_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_true_test,
    y_pred_test,
    average="micro",
    zero_division=0
)


# ------------------------------------------------------------
# 9. Per-disease threshold metrics
# ------------------------------------------------------------

for i, disease in enumerate(disease_cols):

    y_true = y_true_test[:, i]
    y_pred = y_pred_test[:, i]

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    per_disease_results[i].update({
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


per_disease_df = pd.DataFrame(
    per_disease_results
)


# ------------------------------------------------------------
# 10. FINAL BASELINE SUMMARY
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("FINAL DENSENET-121 BASELINE — OFFICIAL TEST SET")
print("=" * 70)

print(f"Test images:             {len(y_true_test):,}")
print(f"Number of labels:        {y_true_test.shape[1]}")

print("\n--- RANKING METRICS ---")
print(f"Macro ROC-AUC:           {macro_roc_auc:.4f}")
print(f"Micro ROC-AUC:           {micro_roc_auc:.4f}")
print(f"Macro PR-AUC:            {macro_pr_auc:.4f}")
print(f"Micro PR-AUC:            {micro_pr_auc:.4f}")

print("\n--- THRESHOLD = 0.50 ---")
print(f"Exact-match accuracy:    {exact_match_accuracy:.4f}")
print(f"Label-wise accuracy:     {label_wise_accuracy:.4f}")
print(f"Hamming loss:            {hamming:.4f}")

print("\n--- MACRO ---")
print(f"Precision:               {macro_precision:.4f}")
print(f"Recall:                  {macro_recall:.4f}")
print(f"F1:                      {macro_f1:.4f}")

print("\n--- MICRO ---")
print(f"Precision:               {micro_precision:.4f}")
print(f"Recall:                  {micro_recall:.4f}")
print(f"F1:                      {micro_f1:.4f}")

print("=" * 70)


# ------------------------------------------------------------
# 11. PER-DISEASE RESULTS
# ------------------------------------------------------------

print("\nPER-DISEASE TEST RESULTS:\n")

display(
    per_disease_df.round(4)
)


# ------------------------------------------------------------
# 12. Save FINAL BASELINE RESULTS
# ------------------------------------------------------------

summary_df = pd.DataFrame([{
    "Model": "DenseNet-121",
    "Best Epoch": checkpoint["epoch"],
    "Test Images": len(y_true_test),
    "Macro ROC-AUC": macro_roc_auc,
    "Micro ROC-AUC": micro_roc_auc,
    "Macro PR-AUC": macro_pr_auc,
    "Micro PR-AUC": micro_pr_auc,
    "Exact Match Accuracy": exact_match_accuracy,
    "Label-wise Accuracy": label_wise_accuracy,
    "Hamming Loss": hamming,
    "Macro Precision": macro_precision,
    "Macro Recall": macro_recall,
    "Macro F1": macro_f1,
    "Micro Precision": micro_precision,
    "Micro Recall": micro_recall,
    "Micro F1": micro_f1
}])

summary_df.to_csv(
    "/kaggle/working/densenet121_final_test_baseline.csv",
    index=False
)

per_disease_df.to_csv(
    "/kaggle/working/densenet121_final_test_per_disease.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/densenet121_final_test_baseline.csv")
print("/kaggle/working/densenet121_final_test_per_disease.csv")

Loaded best DenseNet-121 checkpoint
Best epoch: 2
Best validation loss: 0.1307


FINAL TEST — DenseNet-121:   0%|          | 0/400 [00:00<?, ?it/s]


Prediction shapes:
y_true: (25596, 14)
y_prob: (25596, 14)


FINAL DENSENET-121 BASELINE — OFFICIAL TEST SET
Test images:             25,596
Number of labels:        14

--- RANKING METRICS ---
Macro ROC-AUC:           0.8006
Micro ROC-AUC:           0.8523
Macro PR-AUC:            0.2506
Micro PR-AUC:            0.3516

--- THRESHOLD = 0.50 ---
Exact-match accuracy:    0.3758
Label-wise accuracy:     0.9250
Hamming loss:            0.0750

--- MACRO ---
Precision:               0.3253
Recall:                  0.0837
F1:                      0.1209

--- MICRO ---
Precision:               0.5213
Recall:                  0.1468
F1:                      0.2291

PER-DISEASE TEST RESULTS:



,Disease,ROC-AUC,PR-AUC,Positive Cases,Prevalence,Precision,Recall,F1
0,Atelectasis,0.7713,0.3364,3279,0.1281,0.5386,0.0765,0.1340
1,Cardiomegaly,0.8884,0.3368,1069,0.0418,0.5679,0.0861,0.1495
2,Consolidation,0.7409,0.1586,1815,0.0709,0.0000,0.0000,0.0000
3,Edema,0.8359,0.1546,925,0.0361,0.3636,0.0043,0.0085
4,Effusion,0.8282,0.5143,4658,0.1820,0.5784,0.3697,0.4511
5,Emphysema,0.8966,0.3825,1093,0.0427,0.5250,0.2306,0.3204
6,Fibrosis,0.8207,0.0936,435,0.0170,0.0000,0.0000,0.0000
7,Hernia,0.8466,0.0616,86,0.0034,0.0000,0.0000,0.0000
8,Infiltration,0.7010,0.3965,6112,0.2388,0.4591,0.2102,0.2884
9,Mass,0.8060,0.2946,1748,0.0683,0.5125,0.0469,0.0860



Saved:
/kaggle/working/densenet121_final_test_baseline.csv
/kaggle/working/densenet121_final_test_per_disease.csv


In [36]:
import os

for f in [
    "/kaggle/working/best_densenet121.pth",
    "/kaggle/working/densenet121_final_test_baseline.csv",
    "/kaggle/working/densenet121_final_test_per_disease.csv"
]:
    print(f, "→", os.path.getsize(f) / (1024**2), "MB")

/kaggle/working/best_densenet121.pth → 80.5603380203247 MB
/kaggle/working/densenet121_final_test_baseline.csv → 0.00046539306640625 MB
/kaggle/working/densenet121_final_test_per_disease.csv → 0.0015926361083984375 MB


In [37]:
import zipfile
import os

files_to_save = [
    "/kaggle/working/best_densenet121.pth",
    "/kaggle/working/best_resnet18.pth",
    "/kaggle/working/densenet121_final_test_baseline.csv",
    "/kaggle/working/densenet121_final_test_per_disease.csv"
]

zip_path = "/kaggle/working/mipaas_model_baselines.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_save:
        if os.path.exists(f):
            z.write(f, arcname=os.path.basename(f))
            print("Added:", os.path.basename(f))
        else:
            print("MISSING:", f)

print("\nCreated:", zip_path)
print(
    "Size:",
    os.path.getsize(zip_path) / (1024**2),
    "MB"
)

Added: best_densenet121.pth
Added: best_resnet18.pth
Added: densenet121_final_test_baseline.csv
Added: densenet121_final_test_per_disease.csv

Created: /kaggle/working/mipaas_model_baselines.zip
Size: 189.90171909332275 MB


In [38]:
import os

for f in [
    "/kaggle/working/best_densenet121.pth",
    "/kaggle/working/best_resnet18.pth",
    "/kaggle/working/MIPaaS_baseline_models.zip"
]:
    if os.path.exists(f):
        print("✅", os.path.basename(f),
              f"{os.path.getsize(f)/(1024**2):.2f} MB")
    else:
        print("❌ MISSING:", os.path.basename(f))

✅ best_densenet121.pth 80.56 MB
✅ best_resnet18.pth 128.04 MB
❌ MISSING: MIPaaS_baseline_models.zip


In [3]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [5]:
import os
import glob

print("Searching for baseline model and dataset files...\n")

pth_files = glob.glob(
    "/kaggle/input/**/*.pth",
    recursive=True
)

csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

print("MODEL FILES:")
for f in pth_files:
    print(" ", f)

print("\nIMPORTANT CSV FILES:")
for f in csv_files:
    if any(x in os.path.basename(f).lower()
           for x in ["val_df", "train_df", "official_test", "shard_index"]):
        print(" ", f)

Searching for baseline model and dataset files...



KeyboardInterrupt: 

In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models

# ============================================================
# PATHS
# ============================================================

BASELINE_ROOT = "/kaggle/input/datasets/shreyagupta1181/mipass-baseline-data"

MODEL_PATH = os.path.join(
    BASELINE_ROOT,
    "best_densenet121.pth"
)

VAL_PATH = os.path.join(
    BASELINE_ROOT,
    "val_df.csv"
)

SHARD_INDEX_PATH = (
    "/kaggle/input/datasets/shreyagupta1181/"
    "mipaas-full-preprocessed/shards/shard_index.csv"
)

SHARD_DIR = (
    "/kaggle/input/datasets/shreyagupta1181/"
    "mipaas-full-preprocessed/shards"
)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ============================================================
# CHECK FILES
# ============================================================

print("\nBaseline files:")
for f in os.listdir(BASELINE_ROOT):
    print(" ", f)

print("\nModel exists:", os.path.exists(MODEL_PATH))
print("Validation CSV exists:", os.path.exists(VAL_PATH))
print("Shard index exists:", os.path.exists(SHARD_INDEX_PATH))


# ============================================================
# DISEASE LABELS
# ============================================================

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]


# ============================================================
# LOAD VALIDATION DATA
# ============================================================

val_df = pd.read_csv(VAL_PATH)

print("\nValidation shape:", val_df.shape)


# ============================================================
# LOAD SHARD INDEX
# ============================================================

shard_index = pd.read_csv(
    SHARD_INDEX_PATH
)

print("Shard index:", shard_index.shape)


# ============================================================
# MAP IMAGE → SHARD
# ============================================================

val_df["npy_name"] = (
    val_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

val_df = val_df.merge(
    shard_index,
    left_on="npy_name",
    right_on="file",
    how="left",
    validate="one_to_one"
)

missing = val_df["shard"].isna().sum()

print("Missing mappings:", missing)

if missing > 0:
    raise ValueError(
        f"{missing} validation images have no shard mapping."
    )

val_df["shard"] = val_df["shard"].astype(int)
val_df["row"] = val_df["row"].astype(int)


# ============================================================
# DATASET
# ============================================================

class ShardDataset(Dataset):

    def __init__(self, df, shard_dir):
        self.df = df.reset_index(drop=True)
        self.shard_dir = shard_dir
        self.shards = {}

    def __len__(self):
        return len(self.df)

    def get_shard(self, shard_id):

        if shard_id not in self.shards:

            path = os.path.join(
                self.shard_dir,
                f"shard_{shard_id:02d}.npy"
            )

            self.shards[shard_id] = np.load(
                path,
                mmap_mode="r"
            )

        return self.shards[shard_id]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        shard_id = int(row["shard"])
        row_id = int(row["row"])

        image = self.get_shard(
            shard_id
        )[row_id]

        image = torch.from_numpy(
            image.astype(np.float32)
        ) / 255.0

        image = image.unsqueeze(0)

        target = torch.tensor(
            row[disease_cols].values.astype(np.float32)
        )

        return image, target


val_dataset = ShardDataset(
    val_df,
    SHARD_DIR
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("\nValidation dataset:", len(val_dataset))
print("Validation batches:", len(val_loader))


# ============================================================
# RESTORE DENSENET-121
# ============================================================

model = models.densenet121(
    weights=None
)

# Change RGB → grayscale
old_conv = model.features.conv0

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

model.features.conv0 = new_conv

# Change classifier → 14 diseases
model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("\n========================================")
print("DenseNet-121 restored successfully!")
print("Best epoch:", checkpoint["epoch"])
print(
    f"Best validation loss: "
    f"{checkpoint['val_loss']:.4f}"
)
print("========================================")

Device: cpu

Baseline files:
  densenet121_final_test_baseline.csv
  best_resnet18.pth
  densenet121_final_test_per_disease.csv
  best_densenet121.pth

Model exists: True
Validation CSV exists: False
Shard index exists: True


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/shreyagupta1181/mipass-baseline-data/val_df.csv'

In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models


# ============================================================
# PATHS
# ============================================================

BASELINE_ROOT = (
    "/kaggle/input/datasets/shreyagupta1181/"
    "mipass-baseline-data"
)

# Model is in baseline dataset
MODEL_PATH = os.path.join(
    BASELINE_ROOT,
    "best_densenet121.pth"
)

# Validation data is in original processed dataset
DATA_ROOT = (
    "/kaggle/input/datasets/shreyagupta1181/"
    "mipaas-full-preprocessed"
)

VAL_PATH = os.path.join(
    DATA_ROOT,
    "split",
    "val_df.csv"
)

SHARD_INDEX_PATH = os.path.join(
    DATA_ROOT,
    "shards",
    "shard_index.csv"
)

SHARD_DIR = os.path.join(
    DATA_ROOT,
    "shards"
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# CHECK FILES
# ============================================================

print("\nChecking files...")

print(
    "DenseNet checkpoint:",
    os.path.exists(MODEL_PATH)
)

print(
    "Validation CSV:",
    os.path.exists(VAL_PATH)
)

print(
    "Shard index:",
    os.path.exists(SHARD_INDEX_PATH)
)


# ============================================================
# DISEASE LABELS
# ============================================================

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]


# ============================================================
# LOAD VALIDATION DATA
# ============================================================

val_df = pd.read_csv(VAL_PATH)

print("\nValidation dataframe:")
print(val_df.shape)


# ============================================================
# LOAD SHARD INDEX
# ============================================================

shard_index = pd.read_csv(
    SHARD_INDEX_PATH
)

print("Shard index:")
print(shard_index.shape)


# ============================================================
# MAP VALIDATION IMAGES → SHARD + ROW
# ============================================================

val_df["npy_name"] = (
    val_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

val_df = val_df.merge(
    shard_index,
    left_on="npy_name",
    right_on="file",
    how="left",
    validate="one_to_one"
)

missing = val_df["shard"].isna().sum()

print("\nMissing validation mappings:", missing)

if missing > 0:
    raise ValueError(
        f"{missing} validation images have no shard mapping."
    )

val_df["shard"] = val_df["shard"].astype(int)
val_df["row"] = val_df["row"].astype(int)


# ============================================================
# VALIDATION DATASET
# ============================================================

class ShardDataset(Dataset):

    def __init__(self, df, shard_dir):
        self.df = df.reset_index(drop=True)
        self.shard_dir = shard_dir
        self.shards = {}

    def __len__(self):
        return len(self.df)

    def get_shard(self, shard_id):

        if shard_id not in self.shards:

            path = os.path.join(
                self.shard_dir,
                f"shard_{shard_id:02d}.npy"
            )

            self.shards[shard_id] = np.load(
                path,
                mmap_mode="r"
            )

        return self.shards[shard_id]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        shard_id = int(row["shard"])
        row_id = int(row["row"])

        image = self.get_shard(
            shard_id
        )[row_id]

        image = torch.from_numpy(
            image.astype(np.float32)
        ) / 255.0

        image = image.unsqueeze(0)

        target = torch.tensor(
            row[disease_cols]
            .values
            .astype(np.float32)
        )

        return image, target


val_dataset = ShardDataset(
    val_df,
    SHARD_DIR
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("\nValidation dataset:", len(val_dataset))
print("Validation batches:", len(val_loader))


# ============================================================
# RECREATE DENSENET-121 ARCHITECTURE
# ============================================================

model = models.densenet121(
    weights=None
)

# RGB → grayscale
old_conv = model.features.conv0

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

model.features.conv0 = new_conv

# 1000 ImageNet outputs → 14 disease outputs
model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)


# ============================================================
# LOAD SAVED BEST MODEL
# ============================================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 50)
print("DenseNet-121 restored successfully!")
print("Best epoch:", checkpoint["epoch"])
print(
    f"Best validation loss: "
    f"{checkpoint['val_loss']:.4f}"
)
print("=" * 50)

Device: cpu

Checking files...
DenseNet checkpoint: True
Validation CSV: True
Shard index: True

Validation dataframe:
(8536, 27)
Shard index:
(112120, 3)

Missing validation mappings: 0

Validation dataset: 8536
Validation batches: 134

DenseNet-121 restored successfully!
Best epoch: 2
Best validation loss: 0.1307


In [8]:
import numpy as np
import torch
from tqdm.auto import tqdm

model.eval()

val_targets = []
val_probs = []

with torch.no_grad():

    for images, targets in tqdm(
        val_loader,
        desc="Generating validation predictions"
    ):

        images = images.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        val_targets.append(
            targets.numpy()
        )

        val_probs.append(
            probs.cpu().numpy()
        )


y_val = np.concatenate(
    val_targets,
    axis=0
)

y_prob_val = np.concatenate(
    val_probs,
    axis=0
)

print("\nValidation predictions generated:")
print("y_val:", y_val.shape)
print("y_prob_val:", y_prob_val.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Generating validation predictions:   0%|          | 0/134 [00:00<?, ?it/s]


Validation predictions generated:
y_val: (8536, 14)
y_prob_val: (8536, 14)


In [9]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

best_thresholds = {}

for i, disease in enumerate(disease_cols):

    y_true = y_val[:, i]
    y_prob = y_prob_val[:, i]

    best_f1 = -1
    best_threshold = 0.5
    best_precision = 0.0
    best_recall = 0.0

    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold

            best_precision = precision_score(
                y_true,
                y_pred,
                zero_division=0
            )

            best_recall = recall_score(
                y_true,
                y_pred,
                zero_division=0
            )

    best_thresholds[disease] = best_threshold

    threshold_results.append({
        "Disease": disease,
        "Best Threshold": best_threshold,
        "Validation Precision": best_precision,
        "Validation Recall": best_recall,
        "Validation F1": best_f1
    })


threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.round(4)
)

,Disease,Best Threshold,Validation Precision,Validation Recall,Validation F1
0,Atelectasis,0.21,0.3530,0.4777,0.4060
1,Cardiomegaly,0.14,0.2703,0.3797,0.3158
2,Consolidation,0.12,0.2011,0.2724,0.2314
3,Edema,0.13,0.1646,0.3120,0.2155
4,Effusion,0.32,0.4868,0.6146,0.5433
5,Emphysema,0.15,0.3056,0.3113,0.3084
6,Fibrosis,0.12,0.2114,0.1857,0.1977
7,Hernia,0.05,0.0000,0.0000,0.0000
8,Infiltration,0.21,0.3001,0.4946,0.3735
9,Mass,0.12,0.3591,0.3711,0.3650


In [10]:
import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

# ============================================================
# OFFICIAL TEST DATA
# ============================================================

TEST_PATH = os.path.join(
    DATA_ROOT,
    "metadata",
    "official_test.csv"
)

test_df = pd.read_csv(TEST_PATH)

print("Official test dataframe:", test_df.shape)

# Map images to shards
test_df["npy_name"] = (
    test_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

test_df = test_df.merge(
    shard_index,
    left_on="npy_name",
    right_on="file",
    how="left",
    validate="one_to_one"
)

missing = test_df["shard"].isna().sum()

print("Missing test mappings:", missing)

if missing > 0:
    raise ValueError(
        f"{missing} test images have no shard mapping."
    )

test_df["shard"] = test_df["shard"].astype(int)
test_df["row"] = test_df["row"].astype(int)


# ============================================================
# TEST DATASET
# ============================================================

test_dataset = ShardDataset(
    test_df,
    SHARD_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Test dataset:", len(test_dataset))
print("Test batches:", len(test_loader))

Official test dataframe: (25596, 27)
Missing test mappings: 0
Test dataset: 25596
Test batches: 400


In [11]:
# ============================================================
# GENERATE TEST PROBABILITIES
# ============================================================

test_targets = []
test_probs = []

model.eval()

with torch.no_grad():

    for images, targets in tqdm(
        test_loader,
        desc="Generating official test predictions"
    ):

        images = images.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        test_targets.append(
            targets.cpu().numpy()
        )

        test_probs.append(
            probs.cpu().numpy()
        )


y_true_test = np.concatenate(
    test_targets,
    axis=0
).astype(int)

y_prob_test = np.concatenate(
    test_probs,
    axis=0
)

print("\nTest predictions:")
print("y_true_test:", y_true_test.shape)
print("y_prob_test:", y_prob_test.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Generating official test predictions:   0%|          | 0/400 [00:00<?, ?it/s]


Test predictions:
y_true_test: (25596, 14)
y_prob_test: (25596, 14)


In [12]:
# ============================================================
# APPLY VALIDATION-LOCKED THRESHOLDS TO OFFICIAL TEST
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Apply the thresholds selected on VALIDATION
# ------------------------------------------------------------

# IMPORTANT:
# best_thresholds came ONLY from validation.
# Do NOT optimize them again on test.

y_pred_test_opt = np.zeros_like(y_prob_test, dtype=np.int32)

for i, col in enumerate(disease_cols):
    y_pred_test_opt[:, i] = (
        y_prob_test[:, i] >= best_thresholds[col]
    ).astype(np.int32)


# ------------------------------------------------------------
# 2. Overall metrics
# ------------------------------------------------------------

exact_match_acc = accuracy_score(
    y_true_test,
    y_pred_test_opt
)

label_wise_acc = (y_true_test == y_pred_test_opt).mean()

hamming = hamming_loss(
    y_true_test,
    y_pred_test_opt
)

macro_precision = precision_score(
    y_true_test,
    y_pred_test_opt,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true_test,
    y_pred_test_opt,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true_test,
    y_pred_test_opt,
    average="macro",
    zero_division=0
)

micro_precision = precision_score(
    y_true_test,
    y_pred_test_opt,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_true_test,
    y_pred_test_opt,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_true_test,
    y_pred_test_opt,
    average="micro",
    zero_division=0
)


# ------------------------------------------------------------
# 3. Print results
# ------------------------------------------------------------

print("=" * 65)
print("DENSENET-121 — OPTIMIZED PER-CLASS THRESHOLD TEST RESULTS")
print("=" * 65)

print(f"Exact-match Accuracy : {exact_match_acc:.4f}")
print(f"Label-wise Accuracy  : {label_wise_acc:.4f}")
print(f"Hamming Loss         : {hamming:.4f}")

print("\n--- Macro ---")
print(f"Precision             : {macro_precision:.4f}")
print(f"Recall                : {macro_recall:.4f}")
print(f"F1                    : {macro_f1:.4f}")

print("\n--- Micro ---")
print(f"Precision             : {micro_precision:.4f}")
print(f"Recall                : {micro_recall:.4f}")
print(f"F1                    : {micro_f1:.4f}")

print("=" * 65)

DENSENET-121 — OPTIMIZED PER-CLASS THRESHOLD TEST RESULTS
Exact-match Accuracy : 0.2187
Label-wise Accuracy  : 0.8810
Hamming Loss         : 0.1190

--- Macro ---
Precision             : 0.2583
Recall                : 0.3731
F1                    : 0.2993

--- Micro ---
Precision             : 0.3234
Recall                : 0.5194
F1                    : 0.3986


In [13]:
# ============================================================
# PER-DISEASE TEST METRICS WITH VALIDATION-LOCKED THRESHOLDS
# ============================================================

per_disease_results = []

for i, disease in enumerate(disease_cols):

    precision = precision_score(
        y_true_test[:, i],
        y_pred_test_opt[:, i],
        zero_division=0
    )

    recall = recall_score(
        y_true_test[:, i],
        y_pred_test_opt[:, i],
        zero_division=0
    )

    f1 = f1_score(
        y_true_test[:, i],
        y_pred_test_opt[:, i],
        zero_division=0
    )

    positives = int(y_true_test[:, i].sum())
    predicted_positive = int(y_pred_test_opt[:, i].sum())

    per_disease_results.append({
        "Disease": disease,
        "Threshold": best_thresholds[disease],
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Actual Positives": positives,
        "Predicted Positives": predicted_positive
    })

optimized_per_disease_df = pd.DataFrame(per_disease_results)

display(optimized_per_disease_df)

,Disease,Threshold,Precision,Recall,F1,Actual Positives,Predicted Positives
0,Atelectasis,0.21,0.319936,0.490393,0.387237,3279,5026
1,Cardiomegaly,0.14,0.396175,0.406922,0.401477,1069,1098
2,Consolidation,0.12,0.174584,0.311846,0.223848,1815,3242
3,Edema,0.13,0.178035,0.332973,0.232015,925,1730
4,Effusion,0.32,0.462081,0.633104,0.534239,4658,6382
5,Emphysema,0.15,0.388363,0.525160,0.446519,1093,1478
6,Fibrosis,0.12,0.154762,0.149425,0.152047,435,420
7,Hernia,0.05,0.000000,0.000000,0.000000,86,0
8,Infiltration,0.21,0.342074,0.747055,0.469270,6112,13348
9,Mass,0.12,0.336667,0.404462,0.367464,1748,2100


In [14]:
# ============================================================
# SAVE THRESHOLD + TEST RESULTS
# ============================================================

# Validation threshold table
threshold_df.to_csv(
    "/kaggle/working/densenet121_validation_thresholds.csv",
    index=False
)

# Per-disease optimized test results
optimized_per_disease_df.to_csv(
    "/kaggle/working/densenet121_test_optimized_per_disease.csv",
    index=False
)

# Overall metrics
optimized_test_metrics = pd.DataFrame([{
    "Exact-match Accuracy": exact_match_acc,
    "Label-wise Accuracy": label_wise_acc,
    "Hamming Loss": hamming,
    "Macro Precision": macro_precision,
    "Macro Recall": macro_recall,
    "Macro F1": macro_f1,
    "Micro Precision": micro_precision,
    "Micro Recall": micro_recall,
    "Micro F1": micro_f1
}])

optimized_test_metrics.to_csv(
    "/kaggle/working/densenet121_test_optimized_metrics.csv",
    index=False
)

print("Saved:")
print("/kaggle/working/densenet121_validation_thresholds.csv")
print("/kaggle/working/densenet121_test_optimized_per_disease.csv")
print("/kaggle/working/densenet121_test_optimized_metrics.csv")

Saved:
/kaggle/working/densenet121_validation_thresholds.csv
/kaggle/working/densenet121_test_optimized_per_disease.csv
/kaggle/working/densenet121_test_optimized_metrics.csv


In [15]:
# ============================================================
# CALCULATE CLASS WEIGHTS FOR WEIGHTED BCE
# ============================================================

import torch
import pandas as pd
import numpy as np

# Number of positive and negative samples for each disease
positive_counts = train_df[disease_cols].sum()
negative_counts = len(train_df) - positive_counts

# pos_weight = negatives / positives
pos_weights = negative_counts / positive_counts

# Display everything clearly
class_weight_df = pd.DataFrame({
    "Disease": disease_cols,
    "Positive": positive_counts.values.astype(int),
    "Negative": negative_counts.values.astype(int),
    "Positive_Ratio": (positive_counts / len(train_df)).values,
    "pos_weight": pos_weights.values
})

display(class_weight_df)

NameError: name 'train_df' is not defined

In [16]:
# ============================================================
# RESTORE TRAINING DATAFRAME
# ============================================================

import pandas as pd
import numpy as np
import torch

DATA_ROOT = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

TRAIN_PATH = f"{DATA_ROOT}/split/train_df.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("Train shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

Train shape: (77988, 27)

Columns:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'processed_path']


In [17]:
# ============================================================
# DISEASE LABELS
# ============================================================

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

print("Number of disease classes:", len(disease_cols))

Number of disease classes: 14


In [18]:
# ============================================================
# CALCULATE CLASS WEIGHTS FOR WEIGHTED BCE
# ============================================================

positive_counts = train_df[disease_cols].sum()
negative_counts = len(train_df) - positive_counts

# pos_weight = negative samples / positive samples
pos_weights = negative_counts / positive_counts

class_weight_df = pd.DataFrame({
    "Disease": disease_cols,
    "Positive": positive_counts.values.astype(int),
    "Negative": negative_counts.values.astype(int),
    "Positive_Ratio": (positive_counts / len(train_df)).values,
    "pos_weight": pos_weights.values
})

display(class_weight_df)

,Disease,Positive,Negative,Positive_Ratio,pos_weight
0,Atelectasis,7405,70583,0.094951,9.531803
1,Cardiomegaly,1549,76439,0.019862,49.347321
2,Consolidation,2584,75404,0.033133,29.181115
3,Edema,1253,76735,0.016067,61.241022
4,Effusion,7852,70136,0.100682,8.932247
5,Emphysema,1317,76671,0.016887,58.216401
6,Fibrosis,1111,76877,0.014246,69.196220
7,Hernia,122,77866,0.001564,638.245902
8,Infiltration,12480,65508,0.160025,5.249038
9,Mass,3646,74342,0.046751,20.390016


In [19]:
MAX_POS_WEIGHT = 20.0

pos_weights_clipped = np.minimum(
    pos_weights.values,
    MAX_POS_WEIGHT
)

In [20]:
pos_weights_clipped

array([ 9.53180284, 20.        , 20.        , 20.        ,  8.93224656,
       20.        , 20.        , 20.        ,  5.24903846, 20.        ,
       17.36307982, 20.        , 20.        , 20.        ])

In [21]:
# Convert to PyTorch tensor

pos_weight = torch.tensor(
    pos_weights_clipped,
    dtype=torch.float32,
    device=device
)

print(pos_weight)

tensor([ 9.5318, 20.0000, 20.0000, 20.0000,  8.9322, 20.0000, 20.0000, 20.0000,
         5.2490, 20.0000, 17.3631, 20.0000, 20.0000, 20.0000])


In [2]:
import torch

pos_weight = torch.tensor(
    [
        9.5318, 20.0, 20.0, 20.0,
        8.9322, 20.0, 20.0, 20.0,
        5.2490, 20.0, 20.0, 20.0,
        20.0, 20.0
    ],
    dtype=torch.float32
)

print(pos_weight)

tensor([ 9.5318, 20.0000, 20.0000, 20.0000,  8.9322, 20.0000, 20.0000, 20.0000,
         5.2490, 20.0000, 20.0000, 20.0000, 20.0000, 20.0000])


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision.models import densenet121, DenseNet121_Weights
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
Device: cuda
GPU: Tesla T4


In [4]:
# ============================================================
# PATHS
# ============================================================

DATA_ROOT = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

TRAIN_PATH = f"{DATA_ROOT}/split/train_df.csv"
VAL_PATH = f"{DATA_ROOT}/split/val_df.csv"

SHARD_INDEX_PATH = f"{DATA_ROOT}/shards/shard_index.csv"
SHARD_DIR = f"{DATA_ROOT}/shards"

disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

print("Number of classes:", len(disease_cols))

Number of classes: 14


In [5]:
# ============================================================
# LOAD TRAIN / VALIDATION DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
shard_index = pd.read_csv(SHARD_INDEX_PATH)

print("Raw train:", train_df.shape)
print("Raw val:", val_df.shape)
print("Shard index:", shard_index.shape)


# Convert NIH filename from .png -> .npy
train_df["file"] = (
    train_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

val_df["file"] = (
    val_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)


# Remove old shard/row columns if they somehow exist
for col in ["shard", "row"]:
    if col in train_df.columns:
        train_df = train_df.drop(columns=col)

    if col in val_df.columns:
        val_df = val_df.drop(columns=col)


# Map images to shard locations
train_df = train_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)

val_df = val_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)


print("\nMapped train:", train_df.shape)
print("Mapped val:", val_df.shape)

print(
    "Missing train shard mappings:",
    train_df["shard"].isna().sum()
)

print(
    "Missing val shard mappings:",
    val_df["shard"].isna().sum()
)

Raw train: (77988, 27)
Raw val: (8536, 27)
Shard index: (112120, 3)

Mapped train: (77988, 30)
Mapped val: (8536, 30)
Missing train shard mappings: 0
Missing val shard mappings: 0


In [6]:
# ============================================================
# SHARDED DATASET
# ============================================================

class ShardDataset(Dataset):

    def __init__(self, df, shard_dir):
        self.df = df.reset_index(drop=True)
        self.shard_dir = shard_dir

        self.labels = self.df[disease_cols].values.astype(np.float32)

        # Cache opened memory-mapped shards
        self._shard_cache = {}

    def __len__(self):
        return len(self.df)

    def _get_shard(self, shard_name):

        if shard_name not in self._shard_cache:

            shard_path = os.path.join(
                self.shard_dir,
                shard_name
            )

            self._shard_cache[shard_name] = np.load(
                shard_path,
                mmap_mode="r"
            )

        return self._shard_cache[shard_name]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        shard = self._get_shard(row["shard"])

        image = shard[int(row["row"])]

        # uint8 -> float32 [0,1]
        image = np.asarray(image, dtype=np.float32) / 255.0

        # grayscale channel: H,W -> 1,H,W
        image = torch.from_numpy(image).unsqueeze(0)

        target = torch.from_numpy(self.labels[idx])

        return image, target


train_dataset = ShardDataset(
    train_df,
    SHARD_DIR
)

val_dataset = ShardDataset(
    val_df,
    SHARD_DIR
)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train images: 77988
Validation images: 8536
Train batches: 1219
Validation batches: 134


In [7]:
# ============================================================
# LOCKED MODERATED POSITIVE CLASS WEIGHTS
# ============================================================

pos_weight = torch.tensor(
    [
        9.5318,   # Atelectasis
        20.0000,  # Cardiomegaly
        20.0000,  # Consolidation
        20.0000,  # Edema
        8.9322,   # Effusion
        20.0000,  # Emphysema
        20.0000,  # Fibrosis
        20.0000,  # Hernia
        5.2490,   # Infiltration
        20.0000,  # Mass
        17.3631,  # Nodule
        20.0000,  # Pleural Thickening
        20.0000,  # Pneumonia
        20.0000   # Pneumothorax
    ],
    dtype=torch.float32,
    device=device
)

print("pos_weight:")
print(pos_weight)

print("\nShape:", pos_weight.shape)
print("Device:", pos_weight.device)

pos_weight:
tensor([ 9.5318, 20.0000, 20.0000, 20.0000,  8.9322, 20.0000, 20.0000, 20.0000,
         5.2490, 20.0000, 17.3631, 20.0000, 20.0000, 20.0000], device='cuda:0')

Shape: torch.Size([14])
Device: cuda:0


In [8]:
# ============================================================
# FRESH IMAGENET-PRETRAINED DENSENET-121
# ============================================================

weights = DenseNet121_Weights.DEFAULT

model = densenet121(weights=weights)


# ------------------------------------------------------------
# Convert first convolution from RGB -> grayscale
# ------------------------------------------------------------

old_conv = model.features.conv0

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# Preserve ImageNet information by averaging RGB weights
with torch.no_grad():
    new_conv.weight.copy_(
        old_conv.weight.mean(dim=1, keepdim=True)
    )

model.features.conv0 = new_conv


# ------------------------------------------------------------
# Replace classifier for 14 diseases
# ------------------------------------------------------------

num_features = model.classifier.in_features

model.classifier = nn.Linear(
    num_features,
    len(disease_cols)
)

model = model.to(device)

print("DenseNet-121 ready.")
print("Input channels:", model.features.conv0.in_channels)
print("Output classes:", model.classifier.out_features)
print("Device:", next(model.parameters()).device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 127MB/s] 


DenseNet-121 ready.
Input channels: 1
Output classes: 14
Device: cuda:0


In [9]:
# ============================================================
# WEIGHTED BCE EXPERIMENT
# ============================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

MAX_EPOCHS = 5
PATIENCE = 2

print("Loss:", criterion)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)

Loss: BCEWithLogitsLoss()
Learning rate: 0.0001
Weight decay: 0.0001
Max epochs: 5
Patience: 2


In [11]:
# ============================================================
# FIX SHARD FILENAMES
# ============================================================

# Convert numeric shard IDs into actual .npy filenames
train_df["shard"] = train_df["shard"].apply(
    lambda x: f"shard_{int(x):02d}.npy"
)

val_df["shard"] = val_df["shard"].apply(
    lambda x: f"shard_{int(x):02d}.npy"
)

print("Example train shard values:")
print(train_df["shard"].head().tolist())

print("\nExample val shard values:")
print(val_df["shard"].head().tolist())

Example train shard values:
['shard_00.npy', 'shard_00.npy', 'shard_00.npy', 'shard_00.npy', 'shard_00.npy']

Example val shard values:
['shard_00.npy', 'shard_00.npy', 'shard_00.npy', 'shard_00.npy', 'shard_00.npy']


In [12]:
# ============================================================
# RECREATE DATASETS + DATALOADERS
# ============================================================

train_dataset = ShardDataset(
    train_df,
    SHARD_DIR
)

val_dataset = ShardDataset(
    val_df,
    SHARD_DIR
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 77988
Val: 8536


In [13]:
# ============================================================
# TEST DATALOADER
# ============================================================

images, targets = next(iter(train_loader))

print("Images shape :", images.shape)
print("Targets shape:", targets.shape)
print("Images dtype :", images.dtype)
print("Targets dtype:", targets.dtype)

Images shape : torch.Size([64, 1, 224, 224])
Targets shape: torch.Size([64, 14])
Images dtype : torch.float32
Targets dtype: torch.float32


In [14]:
# ============================================================
# TRAIN WEIGHTED DENSENET-121
# ============================================================

from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

best_val_loss = float("inf")
epochs_without_improvement = 0

BEST_MODEL_PATH = "/kaggle/working/best_densenet121_weighted.pth"

history = []


for epoch in range(MAX_EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]"
    )

    for images, targets in train_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(images)

            loss = criterion(
                logits,
                targets
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_train_loss += (
            loss.item() * images.size(0)
        )

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    train_loss = (
        running_train_loss /
        len(train_dataset)
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]"
        )

        for images, targets in val_bar:

            images = images.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )

            with autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(images)

                loss = criterion(
                    logits,
                    targets
                )

            running_val_loss += (
                loss.item() * images.size(0)
            )


    val_loss = (
        running_val_loss /
        len(val_dataset)
    )


    # ========================================================
    # RECORD HISTORY
    # ========================================================

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })


    print(
        f"\nEpoch {epoch+1}/{MAX_EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Val Loss: {val_loss:.4f}"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "pos_weight": pos_weight.detach().cpu(),
                "disease_cols": disease_cols
            },
            BEST_MODEL_PATH
        )

        print(
            f"✓ Best model saved "
            f"(val loss = {best_val_loss:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            "No improvement. "
            f"Patience: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")
        break


print("\n" + "=" * 60)
print("WEIGHTED DENSENET TRAINING COMPLETE")
print("=" * 60)

print("Best validation loss:", best_val_loss)
print("Saved model:", BEST_MODEL_PATH)

history_df = pd.DataFrame(history)
display(history_df)

Epoch 1/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1/5 | Train Loss: 0.7479 | Val Loss: 0.7084
✓ Best model saved (val loss = 0.7084)


Epoch 2/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 2/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 2/5 | Train Loss: 0.6580 | Val Loss: 0.6869
✓ Best model saved (val loss = 0.6869)


... (3 outputs left out)

Epoch 4/5 [Train]:   0%|          | 0/1219 [00:00<?, ?it/s]

Epoch 4/5 [Val]:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 4/5 | Train Loss: 0.5205 | Val Loss: 0.7340
No improvement. Patience: 2/2

Early stopping triggered.

WEIGHTED DENSENET TRAINING COMPLETE
Best validation loss: 0.6868851166410544
Saved model: /kaggle/working/best_densenet121_weighted.pth


,epoch,train_loss,val_loss
0,1,0.747850,0.708388
1,2,0.657985,0.686885
2,3,0.596926,0.700416
3,4,0.520539,0.734050


In [15]:
# ============================================================
# LOAD BEST WEIGHTED DENSENET CHECKPOINT
# ============================================================

checkpoint = torch.load(
    "/kaggle/working/best_densenet121_weighted.pth",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded weighted DenseNet checkpoint")
print("Best epoch:", checkpoint["epoch"])
print("Best weighted val loss:", checkpoint["val_loss"])

Loaded weighted DenseNet checkpoint
Best epoch: 2
Best weighted val loss: 0.6868851166410544


In [16]:
# ============================================================
# GENERATE VALIDATION PROBABILITIES
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score

y_true_val_weighted = []
y_prob_val_weighted = []

with torch.no_grad():

    val_bar = tqdm(
        val_loader,
        desc="Weighted DenseNet validation predictions"
    )

    for images, targets in val_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(images)

        probs = torch.sigmoid(logits)

        y_true_val_weighted.append(
            targets.numpy()
        )

        y_prob_val_weighted.append(
            probs.cpu().numpy()
        )


y_true_val_weighted = np.vstack(
    y_true_val_weighted
)

y_prob_val_weighted = np.vstack(
    y_prob_val_weighted
)

print("y_true:", y_true_val_weighted.shape)
print("y_prob:", y_prob_val_weighted.shape)

Weighted DenseNet validation predictions:   0%|          | 0/134 [00:00<?, ?it/s]

y_true: (8536, 14)
y_prob: (8536, 14)


In [17]:
# ============================================================
# VALIDATION ROC-AUC / PR-AUC
# ============================================================

weighted_results = []

for i, disease in enumerate(disease_cols):

    roc_auc = roc_auc_score(
        y_true_val_weighted[:, i],
        y_prob_val_weighted[:, i]
    )

    pr_auc = average_precision_score(
        y_true_val_weighted[:, i],
        y_prob_val_weighted[:, i]
    )

    weighted_results.append({
        "Disease": disease,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })


weighted_results_df = pd.DataFrame(
    weighted_results
)

macro_roc_auc_weighted = weighted_results_df["ROC-AUC"].mean()
macro_pr_auc_weighted = weighted_results_df["PR-AUC"].mean()

print("=" * 65)
print("WEIGHTED DENSENET-121 — VALIDATION RESULTS")
print("=" * 65)

print(f"Macro ROC-AUC : {macro_roc_auc_weighted:.4f}")
print(f"Macro PR-AUC  : {macro_pr_auc_weighted:.4f}")

display(weighted_results_df)

WEIGHTED DENSENET-121 — VALIDATION RESULTS
Macro ROC-AUC : 0.8282
Macro PR-AUC  : 0.2201


,Disease,ROC-AUC,PR-AUC
0,Atelectasis,0.801691,0.322592
1,Cardiomegaly,0.895865,0.188105
2,Consolidation,0.819354,0.127633
3,Edema,0.916443,0.169796
4,Effusion,0.905305,0.534300
5,Emphysema,0.876634,0.190688
6,Fibrosis,0.816853,0.128307
7,Hernia,0.889098,0.161901
8,Infiltration,0.705774,0.318706
9,Mass,0.846517,0.286336


In [18]:
# ============================================================
# PER-CLASS F1-OPTIMAL THRESHOLDS
# WEIGHTED DENSENET-121
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd

thresholds_to_test = np.arange(0.05, 0.96, 0.01)

weighted_threshold_results = []
weighted_best_thresholds = {}

for i, disease in enumerate(disease_cols):

    best_threshold = 0.50
    best_f1 = -1
    best_precision = 0
    best_recall = 0

    for threshold in thresholds_to_test:

        y_pred = (
            y_prob_val_weighted[:, i] >= threshold
        ).astype(int)

        precision = precision_score(
            y_true_val_weighted[:, i],
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_true_val_weighted[:, i],
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true_val_weighted[:, i],
            y_pred,
            zero_division=0
        )

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    weighted_best_thresholds[disease] = best_threshold

    weighted_threshold_results.append({
        "Disease": disease,
        "Best Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })


weighted_threshold_df = pd.DataFrame(
    weighted_threshold_results
)

display(weighted_threshold_df)

,Disease,Best Threshold,Precision,Recall,F1
0,Atelectasis,0.78,0.309971,0.490286,0.379814
1,Cardiomegaly,0.82,0.224199,0.398734,0.287016
2,Consolidation,0.69,0.161335,0.432836,0.235056
3,Edema,0.60,0.197917,0.456000,0.276029
4,Effusion,0.73,0.482531,0.633209,0.547696
5,Emphysema,0.81,0.314286,0.311321,0.312796
6,Fibrosis,0.62,0.173611,0.178571,0.176056
7,Hernia,0.73,0.307692,0.210526,0.250000
8,Infiltration,0.49,0.272116,0.559908,0.366240
9,Mass,0.82,0.294212,0.471649,0.362376


In [19]:
# ============================================================
# APPLY WEIGHTED MODEL'S OPTIMAL THRESHOLDS
# ============================================================

y_pred_val_weighted_opt = np.zeros_like(
    y_prob_val_weighted,
    dtype=np.int32
)

for i, disease in enumerate(disease_cols):

    threshold = weighted_best_thresholds[disease]

    y_pred_val_weighted_opt[:, i] = (
        y_prob_val_weighted[:, i] >= threshold
    ).astype(np.int32)


# ============================================================
# OVERALL VALIDATION METRICS
# ============================================================

macro_precision_weighted = precision_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="macro",
    zero_division=0
)

macro_recall_weighted = recall_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="macro",
    zero_division=0
)

macro_f1_weighted = f1_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="macro",
    zero_division=0
)

micro_precision_weighted = precision_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="micro",
    zero_division=0
)

micro_recall_weighted = recall_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="micro",
    zero_division=0
)

micro_f1_weighted = f1_score(
    y_true_val_weighted,
    y_pred_val_weighted_opt,
    average="micro",
    zero_division=0
)

print("=" * 65)
print("WEIGHTED DENSENET — VALIDATION WITH OPTIMAL THRESHOLDS")
print("=" * 65)

print(f"Macro Precision : {macro_precision_weighted:.4f}")
print(f"Macro Recall    : {macro_recall_weighted:.4f}")
print(f"Macro F1        : {macro_f1_weighted:.4f}")

print()

print(f"Micro Precision : {micro_precision_weighted:.4f}")
print(f"Micro Recall    : {micro_recall_weighted:.4f}")
print(f"Micro F1        : {micro_f1_weighted:.4f}")

WEIGHTED DENSENET — VALIDATION WITH OPTIMAL THRESHOLDS
Macro Precision : 0.2585
Macro Recall    : 0.3774
Macro F1        : 0.2955

Micro Precision : 0.2805
Micro Recall    : 0.4691
Micro F1        : 0.3511


In [20]:
# ============================================================
# SAVE WEIGHTED MODEL THRESHOLDS
# ============================================================

weighted_threshold_df.to_csv(
    "/kaggle/working/densenet121_weighted_validation_thresholds.csv",
    index=False
)

print(
    "Saved:",
    "/kaggle/working/densenet121_weighted_validation_thresholds.csv"
)

Saved: /kaggle/working/densenet121_weighted_validation_thresholds.csv


In [21]:
# ============================================================
# OFFICIAL TEST DATASET
# ============================================================

TEST_PATH = f"{DATA_ROOT}/metadata/official_test.csv"

test_df = pd.read_csv(TEST_PATH)

print("Raw test shape:", test_df.shape)

# Map .png -> .npy
test_df["file"] = (
    test_df["Image Index"]
    .str.replace(".png", ".npy", regex=False)
)

# Remove possible old mapping columns
for col in ["shard", "row"]:
    if col in test_df.columns:
        test_df = test_df.drop(columns=col)

# Map to sharded storage
test_df = test_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)

print("Mapped test shape:", test_df.shape)
print(
    "Missing shard mappings:",
    test_df["shard"].isna().sum()
)

# Convert shard number -> filename
test_df["shard"] = test_df["shard"].apply(
    lambda x: f"shard_{int(x):02d}.npy"
)

test_dataset = ShardDataset(
    test_df,
    SHARD_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))

Raw test shape: (25596, 27)
Mapped test shape: (25596, 30)
Missing shard mappings: 0
Test images: 25596
Test batches: 400


In [22]:
# ============================================================
# LOAD BEST WEIGHTED DENSENET
# ============================================================

weighted_checkpoint = torch.load(
    "/kaggle/working/best_densenet121_weighted.pth",
    map_location=device
)

model.load_state_dict(
    weighted_checkpoint["model_state_dict"]
)

model.eval()

print("Loaded weighted DenseNet")
print("Best epoch:", weighted_checkpoint["epoch"])
print("Validation loss:", weighted_checkpoint["val_loss"])

Loaded weighted DenseNet
Best epoch: 2
Validation loss: 0.6868851166410544


In [23]:
# ============================================================
# GENERATE OFFICIAL TEST PREDICTIONS
# ============================================================

y_true_test_weighted = []
y_prob_test_weighted = []

with torch.no_grad():

    test_bar = tqdm(
        test_loader,
        desc="Weighted DenseNet — official test predictions"
    )

    for images, targets in test_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(images)

        probs = torch.sigmoid(logits)

        y_true_test_weighted.append(
            targets.numpy()
        )

        y_prob_test_weighted.append(
            probs.cpu().numpy()
        )


y_true_test_weighted = np.vstack(
    y_true_test_weighted
)

y_prob_test_weighted = np.vstack(
    y_prob_test_weighted
)

print("\nTest predictions:")
print("y_true_test_weighted:", y_true_test_weighted.shape)
print("y_prob_test_weighted:", y_prob_test_weighted.shape)

Weighted DenseNet — official test predictions:   0%|          | 0/400 [00:00<?, ?it/s]


Test predictions:
y_true_test_weighted: (25596, 14)
y_prob_test_weighted: (25596, 14)


In [24]:
# ============================================================
# APPLY LOCKED VALIDATION THRESHOLDS TO OFFICIAL TEST
# ============================================================

y_pred_test_weighted_opt = np.zeros_like(
    y_prob_test_weighted,
    dtype=np.int32
)

for i, disease in enumerate(disease_cols):

    threshold = weighted_best_thresholds[disease]

    y_pred_test_weighted_opt[:, i] = (
        y_prob_test_weighted[:, i] >= threshold
    ).astype(np.int32)

print("Thresholds applied.")
print("Prediction shape:", y_pred_test_weighted_opt.shape)

Thresholds applied.
Prediction shape: (25596, 14)


In [25]:
# ============================================================
# WEIGHTED DENSENET — OPTIMIZED TEST METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

exact_match_acc_weighted = accuracy_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt
)

label_wise_acc_weighted = (
    y_true_test_weighted == y_pred_test_weighted_opt
).mean()

hamming_weighted = hamming_loss(
    y_true_test_weighted,
    y_pred_test_weighted_opt
)

macro_precision_weighted_test = precision_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="macro",
    zero_division=0
)

macro_recall_weighted_test = recall_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="macro",
    zero_division=0
)

macro_f1_weighted_test = f1_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="macro",
    zero_division=0
)

micro_precision_weighted_test = precision_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="micro",
    zero_division=0
)

micro_recall_weighted_test = recall_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="micro",
    zero_division=0
)

micro_f1_weighted_test = f1_score(
    y_true_test_weighted,
    y_pred_test_weighted_opt,
    average="micro",
    zero_division=0
)


# ------------------------------------------------------------
# Ranking metrics
# ------------------------------------------------------------

roc_auc_weighted_per_class = []
pr_auc_weighted_per_class = []

for i in range(len(disease_cols)):

    roc_auc_weighted_per_class.append(
        roc_auc_score(
            y_true_test_weighted[:, i],
            y_prob_test_weighted[:, i]
        )
    )

    pr_auc_weighted_per_class.append(
        average_precision_score(
            y_true_test_weighted[:, i],
            y_prob_test_weighted[:, i]
        )
    )

macro_roc_auc_weighted_test = np.mean(
    roc_auc_weighted_per_class
)

macro_pr_auc_weighted_test = np.mean(
    pr_auc_weighted_per_class
)

# Micro ranking metrics
micro_roc_auc_weighted_test = roc_auc_score(
    y_true_test_weighted.ravel(),
    y_prob_test_weighted.ravel()
)

micro_pr_auc_weighted_test = average_precision_score(
    y_true_test_weighted.ravel(),
    y_prob_test_weighted.ravel()
)


# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("=" * 70)
print("WEIGHTED DENSENET-121 — OFFICIAL TEST RESULTS")
print("=" * 70)

print("\n--- Classification @ Validation-Optimized Thresholds ---")

print(
    f"Exact-match Accuracy : "
    f"{exact_match_acc_weighted:.4f}"
)

print(
    f"Label-wise Accuracy  : "
    f"{label_wise_acc_weighted:.4f}"
)

print(
    f"Hamming Loss         : "
    f"{hamming_weighted:.4f}"
)

print("\n--- Macro ---")

print(
    f"Precision             : "
    f"{macro_precision_weighted_test:.4f}"
)

print(
    f"Recall                : "
    f"{macro_recall_weighted_test:.4f}"
)

print(
    f"F1                    : "
    f"{macro_f1_weighted_test:.4f}"
)

print("\n--- Micro ---")

print(
    f"Precision             : "
    f"{micro_precision_weighted_test:.4f}"
)

print(
    f"Recall                : "
    f"{micro_recall_weighted_test:.4f}"
)

print(
    f"F1                    : "
    f"{micro_f1_weighted_test:.4f}"
)

print("\n--- Ranking Metrics ---")

print(
    f"Macro ROC-AUC         : "
    f"{macro_roc_auc_weighted_test:.4f}"
)

print(
    f"Micro ROC-AUC         : "
    f"{micro_roc_auc_weighted_test:.4f}"
)

print(
    f"Macro PR-AUC          : "
    f"{macro_pr_auc_weighted_test:.4f}"
)

print(
    f"Micro PR-AUC          : "
    f"{micro_pr_auc_weighted_test:.4f}"
)

print("=" * 70)

WEIGHTED DENSENET-121 — OFFICIAL TEST RESULTS

--- Classification @ Validation-Optimized Thresholds ---
Exact-match Accuracy : 0.1866
Label-wise Accuracy  : 0.8625
Hamming Loss         : 0.1375

--- Macro ---
Precision             : 0.2755
Recall                : 0.4311
F1                    : 0.3180

--- Micro ---
Precision             : 0.2896
Recall                : 0.5580
F1                    : 0.3813

--- Ranking Metrics ---
Macro ROC-AUC         : 0.8034
Micro ROC-AUC         : 0.8392
Macro PR-AUC          : 0.2578
Micro PR-AUC          : 0.3066


In [26]:
# ============================================================
# PER-DISEASE WEIGHTED TEST RESULTS
# ============================================================

weighted_test_per_disease = []

for i, disease in enumerate(disease_cols):

    precision = precision_score(
        y_true_test_weighted[:, i],
        y_pred_test_weighted_opt[:, i],
        zero_division=0
    )

    recall = recall_score(
        y_true_test_weighted[:, i],
        y_pred_test_weighted_opt[:, i],
        zero_division=0
    )

    f1 = f1_score(
        y_true_test_weighted[:, i],
        y_pred_test_weighted_opt[:, i],
        zero_division=0
    )

    weighted_test_per_disease.append({
        "Disease": disease,
        "Threshold": weighted_best_thresholds[disease],
        "ROC-AUC": roc_auc_weighted_per_class[i],
        "PR-AUC": pr_auc_weighted_per_class[i],
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Actual Positives": int(
            y_true_test_weighted[:, i].sum()
        ),
        "Predicted Positives": int(
            y_pred_test_weighted_opt[:, i].sum()
        )
    })


weighted_test_per_disease_df = pd.DataFrame(
    weighted_test_per_disease
)

display(weighted_test_per_disease_df)

,Disease,Threshold,ROC-AUC,PR-AUC,Precision,Recall,F1,Actual Positives,Predicted Positives
0,Atelectasis,0.78,0.761494,0.319952,0.317638,0.477280,0.381428,3279,4927
1,Cardiomegaly,0.82,0.880575,0.307949,0.321862,0.446211,0.373971,1069,1482
2,Consolidation,0.69,0.735192,0.154246,0.161907,0.434160,0.235858,1815,4867
3,Edema,0.60,0.841609,0.161720,0.176909,0.410811,0.247315,925,2148
4,Effusion,0.73,0.818656,0.488701,0.433017,0.636325,0.515344,4658,6845
5,Emphysema,0.81,0.903752,0.416914,0.425912,0.523330,0.469622,1093,1343
6,Fibrosis,0.62,0.828730,0.093438,0.155556,0.193103,0.172308,435,540
7,Hernia,0.73,0.922516,0.221067,0.437500,0.162791,0.237288,86,32
8,Infiltration,0.49,0.691426,0.383375,0.310558,0.822480,0.450872,6112,16187
9,Mass,0.82,0.801283,0.281275,0.257931,0.497712,0.339777,1748,3373


In [27]:
# ============================================================
# SAVE WEIGHTED TEST RESULTS
# ============================================================

weighted_test_metrics = pd.DataFrame([{
    "Exact-match Accuracy": exact_match_acc_weighted,
    "Label-wise Accuracy": label_wise_acc_weighted,
    "Hamming Loss": hamming_weighted,
    "Macro Precision": macro_precision_weighted_test,
    "Macro Recall": macro_recall_weighted_test,
    "Macro F1": macro_f1_weighted_test,
    "Micro Precision": micro_precision_weighted_test,
    "Micro Recall": micro_recall_weighted_test,
    "Micro F1": micro_f1_weighted_test,
    "Macro ROC-AUC": macro_roc_auc_weighted_test,
    "Micro ROC-AUC": micro_roc_auc_weighted_test,
    "Macro PR-AUC": macro_pr_auc_weighted_test,
    "Micro PR-AUC": micro_pr_auc_weighted_test
}])

weighted_test_metrics.to_csv(
    "/kaggle/working/densenet121_weighted_test_metrics.csv",
    index=False
)

weighted_test_per_disease_df.to_csv(
    "/kaggle/working/densenet121_weighted_test_per_disease.csv",
    index=False
)

weighted_threshold_df.to_csv(
    "/kaggle/working/densenet121_weighted_validation_thresholds.csv",
    index=False
)

print("Saved:")
print("✓ densenet121_weighted_test_metrics.csv")
print("✓ densenet121_weighted_test_per_disease.csv")
print("✓ densenet121_weighted_validation_thresholds.csv")

Saved:
✓ densenet121_weighted_test_metrics.csv
✓ densenet121_weighted_test_per_disease.csv
✓ densenet121_weighted_validation_thresholds.csv


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Using: cuda


In [4]:
import os
import pandas as pd

BASE_DIR = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

TRAIN_CSV = os.path.join(BASE_DIR, "split", "train_df.csv")
VAL_CSV   = os.path.join(BASE_DIR, "split", "val_df.csv")
TEST_CSV  = os.path.join(BASE_DIR, "metadata", "official_test.csv")

SHARD_DIR = os.path.join(BASE_DIR, "shards")

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("TRAIN COLUMNS:")
print(train_df.columns.tolist())

print("\nVAL COLUMNS:")
print(val_df.columns.tolist())

print("\nTEST COLUMNS:")
print(test_df.columns.tolist())

print("\nShapes:")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

TRAIN COLUMNS:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'processed_path']

VAL COLUMNS:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'processed_path']

TEST COLUMNS:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]

In [5]:
print("TRAIN processed_path examples:")
print(train_df["processed_path"].head(5).tolist())

print("\nVAL processed_path examples:")
print(val_df["processed_path"].head(5).tolist())

print("\nTEST processed_path examples:")
print(test_df["processed_path"].head(5).tolist())

TRAIN processed_path examples:
['/kaggle/working/mipaas_full/processed_npy/00000001_000.npy', '/kaggle/working/mipaas_full/processed_npy/00000001_001.npy', '/kaggle/working/mipaas_full/processed_npy/00000001_002.npy', '/kaggle/working/mipaas_full/processed_npy/00000002_000.npy', '/kaggle/working/mipaas_full/processed_npy/00000004_000.npy']

VAL processed_path examples:
['/kaggle/working/mipaas_full/processed_npy/00000008_000.npy', '/kaggle/working/mipaas_full/processed_npy/00000008_001.npy', '/kaggle/working/mipaas_full/processed_npy/00000008_002.npy', '/kaggle/working/mipaas_full/processed_npy/00000038_000.npy', '/kaggle/working/mipaas_full/processed_npy/00000038_001.npy']

TEST processed_path examples:
['/kaggle/working/mipaas_full/processed_npy/00000003_000.npy', '/kaggle/working/mipaas_full/processed_npy/00000003_001.npy', '/kaggle/working/mipaas_full/processed_npy/00000003_002.npy', '/kaggle/working/mipaas_full/processed_npy/00000003_003.npy', '/kaggle/working/mipaas_full/processe

In [6]:
print("\nUnique processed_path extensions:")
print(
    train_df["processed_path"]
    .astype(str)
    .str.split(".")
    .str[-1]
    .value_counts()
)


Unique processed_path extensions:
processed_path
npy    77988
Name: count, dtype: int64


In [7]:
import os
import pandas as pd

BASE_DIR = "/kaggle/input/datasets/shreyagupta1181/mipaas-full-preprocessed"

TRAIN_CSV = os.path.join(BASE_DIR, "split", "train_df.csv")
VAL_CSV   = os.path.join(BASE_DIR, "split", "val_df.csv")
TEST_CSV  = os.path.join(BASE_DIR, "metadata", "official_test.csv")

SHARD_DIR = os.path.join(BASE_DIR, "shards")
SHARD_INDEX_CSV = os.path.join(SHARD_DIR, "shard_index.csv")

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

shard_index = pd.read_csv(SHARD_INDEX_CSV)

print("Shard index columns:")
print(shard_index.columns.tolist())

print("\nShard index shape:", shard_index.shape)
print("\nFirst rows:")
display(shard_index.head())

print("\nTrain:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Shard index columns:
['file', 'shard', 'row']

Shard index shape: (112120, 3)

First rows:


,file,shard,row
0,00000001_000.npy,0,0
1,00000001_001.npy,0,1
2,00000001_002.npy,0,2
3,00000002_000.npy,0,3
4,00000003_000.npy,0,4



Train: (77988, 27)
Val: (8536, 27)
Test: (25596, 27)


In [8]:
# Create filename column compatible with shard_index
for df in [train_df, val_df, test_df]:
    df["file"] = df["Image Index"].astype(str)

# Merge shard location into each split
train_df = train_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)

val_df = val_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)

test_df = test_df.merge(
    shard_index,
    on="file",
    how="left",
    validate="one_to_one"
)

# Convert shard number to actual filename
for df in [train_df, val_df, test_df]:
    df["shard"] = df["shard"].apply(
        lambda x: f"shard_{int(x):02d}.npy"
    )

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nMissing shard mappings:")
print("Train:", train_df["shard"].isna().sum())
print("Val:", val_df["shard"].isna().sum())
print("Test:", test_df["shard"].isna().sum())

print("\nExample mapping:")
display(
    train_df[["Image Index", "file", "shard", "row"]].head()
)

ValueError: cannot convert float NaN to integer

In [9]:
# Reload the original split CSVs cleanly
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Make sure both sides use strings
for df in [train_df, val_df, test_df]:
    df["file"] = df["Image Index"].astype(str).str.strip()

shard_index["file"] = shard_index["file"].astype(str).str.strip()

# Check matches WITHOUT modifying the dataframes
for name, df in [
    ("Train", train_df),
    ("Val", val_df),
    ("Test", test_df)
]:
    matched = df["file"].isin(set(shard_index["file"]))
    
    print(f"{name}:")
    print("  Total:", len(df))
    print("  Matched:", matched.sum())
    print("  Missing:", (~matched).sum())
    
    if (~matched).sum() > 0:
        print("  Examples of missing:")
        print(df.loc[~matched, "file"].head(10).tolist())
    
    print()

Train:
  Total: 77988
  Matched: 0
  Missing: 77988
  Examples of missing:
['00000001_000.png', '00000001_001.png', '00000001_002.png', '00000002_000.png', '00000004_000.png', '00000005_000.png', '00000005_001.png', '00000005_002.png', '00000005_003.png', '00000005_004.png']

Val:
  Total: 8536
  Matched: 0
  Missing: 8536
  Examples of missing:
['00000008_000.png', '00000008_001.png', '00000008_002.png', '00000038_000.png', '00000038_001.png', '00000038_002.png', '00000038_003.png', '00000038_004.png', '00000038_005.png', '00000038_006.png']

Test:
  Total: 25596
  Matched: 0
  Missing: 25596
  Examples of missing:
['00000003_000.png', '00000003_001.png', '00000003_002.png', '00000003_003.png', '00000003_004.png', '00000003_005.png', '00000003_006.png', '00000003_007.png', '00000013_000.png', '00000013_001.png']



In [10]:
# Reload cleanly
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Convert Image Index (.png) to shard filename (.npy)
for df in [train_df, val_df, test_df]:
    df["file"] = (
        df["Image Index"]
        .astype(str)
        .str.strip()
        .str.replace(".png", ".npy", regex=False)
    )

# Make sure shard index is also clean
shard_index["file"] = shard_index["file"].astype(str).str.strip()

# Merge shard + row information
train_df = train_df.merge(
    shard_index[["file", "shard", "row"]],
    on="file",
    how="left",
    validate="one_to_one"
)

val_df = val_df.merge(
    shard_index[["file", "shard", "row"]],
    on="file",
    how="left",
    validate="one_to_one"
)

test_df = test_df.merge(
    shard_index[["file", "shard", "row"]],
    on="file",
    how="left",
    validate="one_to_one"
)

# Check mapping
print("Missing mappings:")
print("Train:", train_df["shard"].isna().sum())
print("Val:", val_df["shard"].isna().sum())
print("Test:", test_df["shard"].isna().sum())

print("\nExample:")
display(
    train_df[["Image Index", "file", "shard", "row"]].head()
)

Missing mappings:
Train: 0
Val: 0
Test: 0

Example:


,Image Index,file,shard,row
0,00000001_000.png,00000001_000.npy,0,0
1,00000001_001.png,00000001_001.npy,0,1
2,00000001_002.png,00000001_002.npy,0,2
3,00000002_000.png,00000002_000.npy,0,3
4,00000004_000.png,00000004_000.npy,0,12


In [11]:
for df in [train_df, val_df, test_df]:
    df["shard"] = df["shard"].astype(int).apply(
        lambda x: f"shard_{x:02d}.npy"
    )

print(train_df[["file", "shard", "row"]].head())

               file         shard  row
0  00000001_000.npy  shard_00.npy    0
1  00000001_001.npy  shard_00.npy    1
2  00000001_002.npy  shard_00.npy    2
3  00000002_000.npy  shard_00.npy    3
4  00000004_000.npy  shard_00.npy   12


In [12]:
import numpy as np
import torch
from torch.utils.data import Dataset
import os

class ShardDataset(Dataset):
    def __init__(self, df, shard_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.shard_dir = shard_dir
        self.transform = transform

        self.labels = self.df[disease_cols].values.astype(np.float32)
        self._shard_cache = {}

    def __len__(self):
        return len(self.df)

    def _get_shard(self, shard_name):
        if shard_name not in self._shard_cache:
            shard_path = os.path.join(self.shard_dir, shard_name)

            self._shard_cache[shard_name] = np.load(
                shard_path,
                mmap_mode="r"
            )

        return self._shard_cache[shard_name]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        shard = self._get_shard(row["shard"])
        image = shard[int(row["row"])]

        # uint8 image → float32
        image = np.asarray(image, dtype=np.float32)

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image / 255.0).unsqueeze(0)

        target = torch.from_numpy(self.labels[idx])

        return image, target

In [13]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.ToPILImage(),

    transforms.RandomResizedCrop(
        size=224,
        scale=(0.85, 1.0)
    ),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

In [14]:
BATCH_SIZE = 64

train_dataset = ShardDataset(
    train_df,
    SHARD_DIR,
    transform=train_transform
)

val_dataset = ShardDataset(
    val_df,
    SHARD_DIR,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

NameError: name 'disease_cols' is not defined

In [ ]:
disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

print("Number of diseases:", len(disease_cols))
print(disease_cols)

In [15]:
BATCH_SIZE = 64

train_dataset = ShardDataset(
    train_df,
    SHARD_DIR,
    transform=train_transform
)

val_dataset = ShardDataset(
    val_df,
    SHARD_DIR,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

NameError: name 'disease_cols' is not defined

In [16]:
disease_cols = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax"
]

print("Number of diseases:", len(disease_cols))
print(disease_cols)

Number of diseases: 14
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


In [17]:
BATCH_SIZE = 64

train_dataset = ShardDataset(
    train_df,
    SHARD_DIR,
    transform=train_transform
)

val_dataset = ShardDataset(
    val_df,
    SHARD_DIR,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train images: 77988
Validation images: 8536
Train batches: 1219
Validation batches: 134


In [18]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Image range:", images.min().item(), "to", images.max().item())

Images shape: torch.Size([64, 1, 224, 224])
Labels shape: torch.Size([64, 14])
Image dtype: torch.float32
Image range: 0.0 to 1.0


In [19]:
import torch
import torch.nn as nn
from torchvision import models

# Fresh ImageNet-pretrained DenseNet-121
weights = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=weights)

# Convert first convolution from RGB → grayscale
old_conv = model.features.conv0

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

# Average pretrained RGB weights
with torch.no_grad():
    new_conv.weight.copy_(
        old_conv.weight.mean(dim=1, keepdim=True)
    )

model.features.conv0 = new_conv

# Replace classifier for 14 diseases
model.classifier = nn.Linear(
    model.classifier.in_features,
    len(disease_cols)
)

model = model.to(device)

print(model.classifier)
print("Device:", next(model.parameters()).device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 183MB/s]


Linear(in_features=1024, out_features=14, bias=True)
Device: cuda:0


In [20]:
pos_weights = torch.tensor([
    9.531803,
    20.0,
    20.0,
    20.0,
    8.932247,
    20.0,
    20.0,
    20.0,
    5.249038,
    20.0,
    17.363080,
    20.0,
    20.0,
    20.0
], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

print("Positive weights:")
print(pos_weights)

Positive weights:
tensor([ 9.5318, 20.0000, 20.0000, 20.0000,  8.9322, 20.0000, 20.0000, 20.0000,
         5.2490, 20.0000, 17.3631, 20.0000, 20.0000, 20.0000], device='cuda:0')


In [21]:
MAX_EPOCHS = 15
BASE_LR = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = len(train_loader)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup_steps = steps_per_epoch

def lr_lambda(current_step):
    # 1 epoch linear warmup
    if current_step < warmup_steps:
        return 0.1 + 0.9 * (
            current_step / warmup_steps
        )

    # Cosine decay after warmup
    progress = (
        current_step - warmup_steps
    ) / (
        total_steps - warmup_steps
    )

    return 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

print("Steps/epoch:", steps_per_epoch)
print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)
print("Initial LR:", optimizer.param_groups[0]["lr"])

Steps/epoch: 1219
Total steps: 18285
Warmup steps: 1219
Initial LR: 1e-05


In [22]:
scaler = torch.amp.GradScaler("cuda")

print("AMP enabled")

AMP enabled


In [23]:
from sklearn.metrics import roc_auc_score
import numpy as np

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()

    running_loss = 0.0
    total_samples = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Scheduler is stepped after every optimizer update
        scheduler.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    return running_loss / total_samples


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        probs = torch.sigmoid(logits)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        all_probs.append(probs.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    avg_loss = running_loss / total_samples

    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)

    # Calculate ROC-AUC for each disease
    aucs = []

    for i in range(all_targets.shape[1]):
        # Safety check in case a class has only one label
        if len(np.unique(all_targets[:, i])) > 1:
            aucs.append(
                roc_auc_score(
                    all_targets[:, i],
                    all_probs[:, i]
                )
            )

    macro_auc = np.mean(aucs)

    return avg_loss, macro_auc, all_probs, all_targets

In [24]:
from copy import deepcopy

PATIENCE = 3

best_macro_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0

checkpoint_path = "/kaggle/working/best_densenet121_weighted_aug_cosine.pth"

history = []

for epoch in range(1, MAX_EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scheduler,
        scaler
    )

    val_loss, val_macro_auc, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": val_macro_auc,
        "lr": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Val Macro ROC-AUC: {val_macro_auc:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    # Early stopping based on Macro ROC-AUC
    if val_macro_auc > best_macro_auc:
        best_macro_auc = val_macro_auc
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_auc": val_macro_auc
            },
            checkpoint_path
        )

        print("  ✓ New best model saved!")

    else:
        epochs_without_improvement += 1

        print(
            f"  No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

        if epochs_without_improvement >= PATIENCE:
            print("  Early stopping triggered.")
            break

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation Macro ROC-AUC:", best_macro_auc)
print("Checkpoint:", checkpoint_path)

Epoch 01/15 | Train Loss: 0.824118 | Val Loss: 0.740776 | Val Macro ROC-AUC: 0.7960 | LR: 1.00e-04
  ✓ New best model saved!
Epoch 02/15 | Train Loss: 0.711967 | Val Loss: 0.699920 | Val Macro ROC-AUC: 0.8151 | LR: 9.87e-05
  ✓ New best model saved!
Epoch 03/15 | Train Loss: 0.670513 | Val Loss: 0.690504 | Val Macro ROC-AUC: 0.8235 | LR: 9.50e-05
  ✓ New best model saved!
Epoch 04/15 | Train Loss: 0.640076 | Val Loss: 0.692051 | Val Macro ROC-AUC: 0.8221 | LR: 8.91e-05
  No improvement (1/3)
Epoch 05/15 | Train Loss: 0.610533 | Val Loss: 0.695216 | Val Macro ROC-AUC: 0.8240 | LR: 8.12e-05
  ✓ New best model saved!
Epoch 06/15 | Train Loss: 0.573287 | Val Loss: 0.710018 | Val Macro ROC-AUC: 0.8228 | LR: 7.17e-05
  No improvement (1/3)
Epoch 07/15 | Train Loss: 0.532856 | Val Loss: 0.727374 | Val Macro ROC-AUC: 0.8170 | LR: 6.11e-05
  No improvement (2/3)
Epoch 08/15 | Train Loss: 0.489149 | Val Loss: 0.803797 | Val Macro ROC-AUC: 0.8108 | LR: 5.00e-05
  No improvement (3/3)
  Early stop

In [28]:
# Save training history
history_df = pd.DataFrame(history)
history_path = "/kaggle/working/densenet121_weighted_aug_cosine_history.csv"
history_df.to_csv(history_path, index=False)

# Load best checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best checkpoint")
print("Best epoch:", checkpoint["epoch"])
print("Best validation Macro ROC-AUC:", checkpoint["val_macro_auc"])

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [26]:
# Save training history
history_df = pd.DataFrame(history)

history_path = (
    "/kaggle/working/"
    "densenet121_weighted_aug_cosine_history.csv"
)

history_df.to_csv(history_path, index=False)


# Load best checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded best checkpoint")
print("Best epoch:", checkpoint["epoch"])
print(
    "Best validation Macro ROC-AUC:",
    checkpoint["val_macro_auc"]
)

Loaded best checkpoint
Best epoch: 5
Best validation Macro ROC-AUC: 0.8240261306238171


In [27]:
val_probs, val_targets = get_predictions(
    model,
    val_loader
)

print("Validation probabilities:", val_probs.shape)
print("Validation targets:", val_targets.shape)

NameError: name 'get_predictions' is not defined

In [29]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()

    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast("cuda"):
            logits = model(images)

        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(targets.numpy())

    return (
        np.concatenate(all_probs),
        np.concatenate(all_targets)
    )

In [30]:
val_probs, val_targets = get_predictions(
    model,
    val_loader
)

print("Validation probabilities:", val_probs.shape)
print("Validation targets:", val_targets.shape)

Validation probabilities: (8536, 14)
Validation targets: (8536, 14)


In [31]:
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

threshold_grid = np.arange(0.05, 0.951, 0.01)

threshold_results = []

for i, disease in enumerate(disease_cols):

    best_threshold = 0.50
    best_f1 = -1.0

    for threshold in threshold_grid:

        preds = (
            val_probs[:, i] >= threshold
        ).astype(int)

        f1 = f1_score(
            val_targets[:, i],
            preds,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    threshold_results.append({
        "disease": disease,
        "threshold": best_threshold,
        "validation_f1": best_f1
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_path = (
    "/kaggle/working/"
    "densenet121_weighted_aug_cosine_validation_thresholds.csv"
)

threshold_df.to_csv(
    threshold_path,
    index=False
)

display(threshold_df)

,disease,threshold,validation_f1
0,Atelectasis,0.67,0.401738
1,Cardiomegaly,0.86,0.276923
2,Consolidation,0.68,0.239777
3,Edema,0.85,0.246649
4,Effusion,0.74,0.546731
5,Emphysema,0.88,0.300971
6,Fibrosis,0.68,0.174334
7,Hernia,0.68,0.482759
8,Infiltration,0.63,0.356173
9,Mass,0.79,0.399015


In [32]:
optimal_thresholds = threshold_df["threshold"].values

val_preds = (
    val_probs >= optimal_thresholds
).astype(int)

macro_precision = []
macro_recall = []
macro_f1 = []

for i in range(len(disease_cols)):
    from sklearn.metrics import precision_score, recall_score

    macro_precision.append(
        precision_score(
            val_targets[:, i],
            val_preds[:, i],
            zero_division=0
        )
    )

    macro_recall.append(
        recall_score(
            val_targets[:, i],
            val_preds[:, i],
            zero_division=0
        )
    )

    macro_f1.append(
        f1_score(
            val_targets[:, i],
            val_preds[:, i],
            zero_division=0
        )
    )

print("Validation threshold-selected metrics")
print("--------------------------------------")
print("Macro Precision:", np.mean(macro_precision))
print("Macro Recall:   ", np.mean(macro_recall))
print("Macro F1:       ", np.mean(macro_f1))

micro_precision = precision_score(
    val_targets.flatten(),
    val_preds.flatten(),
    zero_division=0
)

micro_recall = recall_score(
    val_targets.flatten(),
    val_preds.flatten(),
    zero_division=0
)

micro_f1 = f1_score(
    val_targets.flatten(),
    val_preds.flatten(),
    zero_division=0
)

print("Micro Precision:", micro_precision)
print("Micro Recall:   ", micro_recall)
print("Micro F1:       ", micro_f1)

Validation threshold-selected metrics
--------------------------------------
Macro Precision: 0.3104348643958207
Macro Recall:    0.3519171460895268
Macro F1:        0.3128008043653499
Micro Precision: 0.3100252737994945
Micro Recall:    0.42209902504301283
Micro F1:        0.35748401198089536


In [33]:
test_dataset = ShardDataset(
    test_df,
    SHARD_DIR,
    transform=val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))

Test images: 25596
Test batches: 400


In [34]:
test_probs, test_targets = get_predictions(
    model,
    test_loader
)

print("Test probabilities:", test_probs.shape)
print("Test targets:", test_targets.shape)

Test probabilities: (25596, 14)
Test targets: (25596, 14)


In [35]:
optimal_thresholds = threshold_df["threshold"].values

test_preds = (
    test_probs >= optimal_thresholds
).astype(int)

print("Predictions shape:", test_preds.shape)

Predictions shape: (25596, 14)


In [36]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss
)

# -----------------------------
# Ranking metrics
# -----------------------------

macro_roc_auc = roc_auc_score(
    test_targets,
    test_probs,
    average="macro"
)

micro_roc_auc = roc_auc_score(
    test_targets,
    test_probs,
    average="micro"
)

macro_pr_auc = average_precision_score(
    test_targets,
    test_probs,
    average="macro"
)

micro_pr_auc = average_precision_score(
    test_targets,
    test_probs,
    average="micro"
)

# -----------------------------
# Threshold-based metrics
# -----------------------------

macro_precision = precision_score(
    test_targets,
    test_preds,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    test_targets,
    test_preds,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    test_targets,
    test_preds,
    average="macro",
    zero_division=0
)

micro_precision = precision_score(
    test_targets,
    test_preds,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    test_targets,
    test_preds,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    test_targets,
    test_preds,
    average="micro",
    zero_division=0
)

exact_match = np.mean(
    np.all(test_targets == test_preds, axis=1)
)

label_accuracy = np.mean(
    test_targets == test_preds
)

ham_loss = hamming_loss(
    test_targets,
    test_preds
)

print("======================================")
print("EXPERIMENT 3 — OFFICIAL TEST RESULTS")
print("======================================")

print(f"Macro ROC-AUC:       {macro_roc_auc:.4f}")
print(f"Micro ROC-AUC:       {micro_roc_auc:.4f}")
print(f"Macro PR-AUC:        {macro_pr_auc:.4f}")
print(f"Micro PR-AUC:        {micro_pr_auc:.4f}")
print()
print(f"Macro Precision:     {macro_precision:.4f}")
print(f"Macro Recall:        {macro_recall:.4f}")
print(f"Macro F1:            {macro_f1:.4f}")
print()
print(f"Micro Precision:     {micro_precision:.4f}")
print(f"Micro Recall:        {micro_recall:.4f}")
print(f"Micro F1:            {micro_f1:.4f}")
print()
print(f"Exact-match Accuracy:{exact_match:.4f}")
print(f"Label-wise Accuracy: {label_accuracy:.4f}")
print(f"Hamming Loss:        {ham_loss:.4f}")

EXPERIMENT 3 — OFFICIAL TEST RESULTS
Macro ROC-AUC:       0.8009
Micro ROC-AUC:       0.8431
Macro PR-AUC:        0.2702
Micro PR-AUC:        0.3119

Macro Precision:     0.3089
Macro Recall:        0.3805
Macro F1:            0.3195

Micro Precision:     0.3163
Micro Recall:        0.4949
Micro F1:            0.3859

Exact-match Accuracy:0.2169
Label-wise Accuracy: 0.8804
Hamming Loss:        0.1196


In [37]:
per_disease_results = []

for i, disease in enumerate(disease_cols):

    y_true = test_targets[:, i]
    y_prob = test_probs[:, i]
    y_pred = test_preds[:, i]

    per_disease_results.append({
        "disease": disease,
        "threshold": optimal_thresholds[i],
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "actual_positives": int(y_true.sum()),
        "predicted_positives": int(y_pred.sum())
    })

per_disease_df = pd.DataFrame(per_disease_results)

display(per_disease_df)

,disease,threshold,ROC_AUC,PR_AUC,precision,recall,F1,actual_positives,predicted_positives
0,Atelectasis,0.67,0.760292,0.312462,0.315683,0.476365,0.379725,3279,4948
1,Cardiomegaly,0.86,0.882776,0.301355,0.355000,0.398503,0.375496,1069,1200
2,Consolidation,0.68,0.736637,0.157942,0.158539,0.499725,0.240711,1815,5721
3,Edema,0.85,0.841780,0.172031,0.182507,0.338378,0.237121,925,1715
4,Effusion,0.74,0.817722,0.491393,0.432545,0.647703,0.518697,4658,6975
5,Emphysema,0.88,0.901706,0.398106,0.389539,0.517841,0.444619,1093,1453
6,Fibrosis,0.68,0.824634,0.106087,0.147099,0.250575,0.185374,435,741
7,Hernia,0.68,0.909849,0.376027,0.685714,0.279070,0.396694,86,35
8,Infiltration,0.63,0.689777,0.387341,0.345691,0.700262,0.462878,6112,12381
9,Mass,0.79,0.805418,0.288383,0.317958,0.406178,0.356694,1748,2233


In [38]:
# Save test metrics
test_metrics = {
    "macro_roc_auc": macro_roc_auc,
    "micro_roc_auc": micro_roc_auc,
    "macro_pr_auc": macro_pr_auc,
    "micro_pr_auc": micro_pr_auc,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "micro_precision": micro_precision,
    "micro_recall": micro_recall,
    "micro_f1": micro_f1,
    "exact_match_accuracy": exact_match,
    "label_wise_accuracy": label_accuracy,
    "hamming_loss": ham_loss
}

test_metrics_df = pd.DataFrame([test_metrics])

metrics_path = (
    "/kaggle/working/"
    "densenet121_weighted_aug_cosine_test_metrics.csv"
)

per_disease_path = (
    "/kaggle/working/"
    "densenet121_weighted_aug_cosine_test_per_disease.csv"
)

test_metrics_df.to_csv(metrics_path, index=False)
per_disease_df.to_csv(per_disease_path, index=False)

print("Saved:")
print(metrics_path)
print(per_disease_path)
print(threshold_path)
print(history_path)
print(checkpoint_path)

Saved:
/kaggle/working/densenet121_weighted_aug_cosine_test_metrics.csv
/kaggle/working/densenet121_weighted_aug_cosine_test_per_disease.csv
/kaggle/working/densenet121_weighted_aug_cosine_validation_thresholds.csv
/kaggle/working/densenet121_weighted_aug_cosine_history.csv
/kaggle/working/best_densenet121_weighted_aug_cosine.pth
